# Acne Lesion Segmentation & Severity Classification
**Dataset:** ACNE04-v2 | **Model:** Hybrid ResNet-50 + Dilated Decoder + AMFM

---
## Urutan Menjalankan
Jalankan setup dan preprocessing secara berurutan, lalu pilih salah satu jalur: legacy Steps 11–17 atau eksperimen flowchart Steps 18–21. Untuk jalur flowchart, lewati training legacy agar tidak melatih model dua kali.

| # | Cell | Waktu |
|---|---|---|
| 1 | Setup GPU & Import | ~30 detik |
| 2 | Mount Google Drive | ~10 detik |
| 3 | Install Dependencies | ~1 menit |
| 4 | Import Python Libraries | ~5 detik |
| 5 | Download Dataset dari Folder Drive | ~3 menit |
| 6 | Extract & Auto-detect Path | ~2 menit |
| 7 | Ekstrak Scripts dari Drive | ~30 detik |
| 8 | Konfigurasi & Validasi Path | ~5 detik |
| 9 | Generate refined masks & manifest | ~5-15 menit |
| 11–17 | Training dan inference legacy | ~2-4 jam |
| 18–21 | Audit foto, 5-fold CV, ablation, final flowchart run (opsional) | beberapa jam hingga beberapa hari |


---
## 1. Setup GPU
> **Aktifkan GPU dulu sebelum menjalankan notebook:**  
> `Runtime → Change runtime type → Hardware accelerator → T4 GPU → Save`

In [5]:
import torch

print('PyTorch version :', torch.__version__)
print('CUDA available  :', torch.cuda.is_available())

if torch.cuda.is_available():
    print('GPU name        :', torch.cuda.get_device_name(0))
    print('VRAM            :', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print()
    print('⚠️  GPU tidak aktif!')
    print('   Aktifkan: Runtime → Change runtime type → T4 GPU')

PyTorch version : 2.11.0+cpu
CUDA available  : False

⚠️  GPU tidak aktif!
   Aktifkan: Runtime → Change runtime type → T4 GPU


---
## 2. Mount Google Drive

In [4]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ Google Drive berhasil di-mount di /content/drive')

Mounted at /content/drive
✅ Google Drive berhasil di-mount di /content/drive


---
## 3. Install Dependencies

In [ ]:
%%capture
!pip install torch torchvision pillow opencv-python-headless numpy matplotlib pandas tqdm scikit-learn gdown

---
## 4. Import Python Libraries
> **Wajib dijalankan sebelum cell apapun di bawah ini.**

In [ ]:
import os
import sys
import json
import shutil
import random
import tarfile
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm

import torch

print('✅ Semua library berhasil diimport')
print(f'   Python  : {sys.version.split()[0]}')
print(f'   PyTorch : {torch.__version__}')
print(f'   Device  : {"cuda" if torch.cuda.is_available() else "cpu"}')

---
## 5. Download Gambar Sumber dari Google Drive

Gambar sumber diambil dari Drive dan dapat berisi seluruh ACNE04 original. Jumlah file dalam folder bukan jumlah sampel v2. Step 6 mengambil JSON resmi v2; Step 8/10 memverifikasi subset 1.204 gambar sebelum augmentasi.

`https://drive.google.com/drive/folders/18yJcHXhzOv7H89t-Lda6phheAicLqMuZ`


In [ ]:
# ══════════════════════════════════════════════════════════════
# Dataset dari folder Google Drive publik
# ══════════════════════════════════════════════════════════════

import gdown

DATASET_DRIVE_FOLDER = 'https://drive.google.com/drive/folders/18yJcHXhzOv7H89t-Lda6phheAicLqMuZ'
DOWNLOAD_DIR = Path('/content/acne_images')
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

gdown.download_folder(
    DATASET_DRIVE_FOLDER,
    output=str(DOWNLOAD_DIR),
    quiet=False,
    use_cookies=False,
)

print('✅ Dataset selesai didownload ke', DOWNLOAD_DIR)


---
## 6. Extract Tar & Auto-detect Path Dataset

Folder Drive dataset berisi archive seperti `Classification.tar` dan `Detection.tar`.
Cell berikut mengekstrak arsip dan mencari folder gambar. Anotasi diambil khusus dari JSON resmi ACNE04-v2 pada commit yang dipin, bukan XML original dalam Detection.tar. File sumber di luar 1.204 nama pada JSON tidak digunakan. Jangan menghapus gambar sumber hanya untuk menyesuaikan jumlah.


In [ ]:
# Extract semua archive tar yang ditemukan di DOWNLOAD_DIR
archive_patterns = ['*.tar', '*.tar.gz', '*.tgz']
processed_archives = set()

while True:
    tar_files = []
    for pattern in archive_patterns:
        tar_files.extend(DOWNLOAD_DIR.rglob(pattern))
    tar_files = sorted(p for p in tar_files if p not in processed_archives)

    if not tar_files:
        break

    for tar_path in tar_files:
        extract_to = tar_path.parent
        print(f'Extracting {tar_path.name} → {extract_to}')
        with tarfile.open(tar_path) as tf:
            tf.extractall(path=extract_to)
        processed_archives.add(tar_path)

if processed_archives:
    print(f'✅ {len(processed_archives)} archive berhasil diekstrak')
else:
    print('Tidak ada file .tar/.tar.gz/.tgz ditemukan — mungkin dataset sudah berupa folder langsung.')

print()
print('Struktur folder setelah download/extract (maks. 120 item):')
all_paths = sorted(DOWNLOAD_DIR.rglob('*'))
for p in all_paths[:120]:
    indent = '  ' * (len(p.relative_to(DOWNLOAD_DIR).parts) - 1)
    print(f'{indent}  {p.name}{"/ " if p.is_dir() else ""}')
if len(all_paths) > 120:
    print(f'... {len(all_paths) - 120} item lain disembunyikan')


In [ ]:
# Auto-detect folder gambar; anotasi wajib JSON resmi ACNE04-v2.

def find_image_dir(root):
    best_dir, best_count = None, 0
    for d in root.rglob('*'):
        if d.is_dir():
            count = len(list(d.glob('*.jpg'))) + len(list(d.glob('*.JPG'))) + len(list(d.glob('*.jpeg'))) + len(list(d.glob('*.JPEG')))
            if count > best_count:
                best_dir, best_count = d, count
    return best_dir

DETECTED_IMAGE_DIR = find_image_dir(DOWNLOAD_DIR)
# V2 strict: jangan auto-fallback ke anotasi XML dataset original.
from urllib.request import urlopen
import hashlib
V2_COMMIT = '5930801fc75fb1a7084c26c1db3095d3e42d1f55'
V2_ANNOT_SHA256 = 'ae03fc23fed0925c254dc4c4c9e55b539bd199b236151cb649857378a43539e9'
V2_ANNOT_URL = f'https://raw.githubusercontent.com/AIpourlapeau/acne04v2/{V2_COMMIT}/Acne04-v2_annotations.json'
DETECTED_ANNOT_JSON = DOWNLOAD_DIR / 'official_acne04v2' / 'Acne04-v2_annotations.json'
if not DETECTED_ANNOT_JSON.exists():
    with urlopen(V2_ANNOT_URL, timeout=60) as response:
        annotation_bytes = response.read()
    if hashlib.sha256(annotation_bytes).hexdigest() != V2_ANNOT_SHA256:
        raise ValueError('Unduhan anotasi v2 gagal verifikasi SHA-256.')
    DETECTED_ANNOT_JSON.parent.mkdir(parents=True, exist_ok=True)
    DETECTED_ANNOT_JSON.write_bytes(annotation_bytes)
if hashlib.sha256(DETECTED_ANNOT_JSON.read_bytes()).hexdigest() != V2_ANNOT_SHA256:
    raise ValueError('File anotasi tersimpan bukan v2 resmi. Periksa file; tidak ditimpa otomatis.')

print('=== AUTO-DETECT RESULT ===')
if DETECTED_IMAGE_DIR:
    n_img = len(list(DETECTED_IMAGE_DIR.glob('*.jpg'))) + len(list(DETECTED_IMAGE_DIR.glob('*.JPG'))) + len(list(DETECTED_IMAGE_DIR.glob('*.jpeg'))) + len(list(DETECTED_IMAGE_DIR.glob('*.JPEG')))
    print(f'✅ Image dir  : {DETECTED_IMAGE_DIR}')
    print(f'   Jumlah img : {n_img}')
else:
    raise FileNotFoundError('Image dir tidak ditemukan di hasil download Drive. Cek isi Classification.tar.')

if DETECTED_ANNOT_JSON:
    if DETECTED_ANNOT_JSON.is_dir():
        print(f'✅ Annotation : {DETECTED_ANNOT_JSON} ({len(list(DETECTED_ANNOT_JSON.glob("*.xml")))} XML)')
    else:
        print(f'✅ Annotation : {DETECTED_ANNOT_JSON}')
else:
    raise FileNotFoundError('Annotation tidak ditemukan. Detection.tar harus berisi JSON atau folder XML Annotations.')


---
## 7. Setup Scripts Project dari Google Drive

Jalankan cell di bawah untuk mengekstrak `MyDrive/acne_project/acne_scripts.zip`.


In [ ]:
# ══════════════════════════════════════════════════════════════
# Scripts dari Google Drive sendiri
#
# Struktur yang wajib ada di Drive:
#   MyDrive/acne_project/acne_scripts.zip
# ══════════════════════════════════════════════════════════════

SCRIPTS_ZIP = Path('/content/drive/MyDrive/acne_project/acne_scripts.zip')
SCRIPTS_ROOT = Path('/content/acne_project')
SCRIPTS_ROOT.mkdir(parents=True, exist_ok=True)

if not SCRIPTS_ZIP.exists():
    raise FileNotFoundError(
        'acne_scripts.zip tidak ditemukan. Upload ke MyDrive/acne_project/acne_scripts.zip'
    )

!unzip -q -o "{SCRIPTS_ZIP}" -d "{SCRIPTS_ROOT}/"

script_candidates = list(SCRIPTS_ROOT.rglob('refine_masks_color.py'))
if not script_candidates:
    raise FileNotFoundError('refine_masks_color.py tidak ditemukan di acne_scripts.zip')

SCRIPTS_DIR = script_candidates[0].parent
print('✅ Scripts dir:', SCRIPTS_DIR)
print('Exists:', SCRIPTS_DIR.exists())


In [ ]:
# Tambahkan scripts ke Python path
for p in [str(SCRIPTS_DIR), str(SCRIPTS_DIR / 'code_dari_claude')]:
    if p not in sys.path:
        sys.path.insert(0, p)

py_files = list(SCRIPTS_DIR.glob('*.py'))
print(f'✅ {len(py_files)} script Python ditemukan di {SCRIPTS_DIR}')
for f in sorted(py_files):
    print(f'   {f.name}')

---
## 8. Konfigurasi Path

> Biasanya tidak perlu diubah jika pakai auto-detect di atas.  
> Ubah manual hanya jika path salah.

In [ ]:
# ============================================================
# KONFIGURASI TERPUSAT — semua path & parameter ada di sini
# ============================================================

# --- INPUT ---
IMAGE_DIR  = DETECTED_IMAGE_DIR    # dari auto-detect
ANNOT_JSON = DETECTED_ANNOT_JSON   # JSON resmi ACNE04-v2 yang dipin; bukan XML original

# Override manual jika auto-detect salah:
# IMAGE_DIR  = Path('/content/acne_images/ACNE04/Classification/small_1024')
# Jangan override ANNOT_JSON ke XML original; audit akan menolaknya.

# --- OUTPUT (Drive, agar tidak hilang saat sesi berakhir) ---
OUTPUT_ROOT = Path('/content/drive/MyDrive/acne_project/output')
MASK_DIR    = OUTPUT_ROOT / 'acne04v2_official1204_letterbox_v1'
TRAIN_DIR   = OUTPUT_ROOT / 'training_runs'
INFER_DIR   = OUTPUT_ROOT / 'predictions'

for d in [OUTPUT_ROOT, MASK_DIR, TRAIN_DIR, INFER_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# --- Device & manifest ---
DEVICE           = 'cuda' if torch.cuda.is_available() else 'cpu'
REFINED_MANIFEST = MASK_DIR / 'refined_manifest.csv'

# --- Script paths ---
REFINE_SCRIPT  = SCRIPTS_DIR / 'refine_masks_color.py'
TRAIN_SCRIPT_A = SCRIPTS_DIR / 'train_proposal_hybrid_resnet50_malds.py'
TRAIN_SCRIPT_B = SCRIPTS_DIR / 'code_dari_claude' / 'train_hybrid_resnet50_staged.py'
INFER_SCRIPT   = SCRIPTS_DIR / 'infer_testing_images.py'

# --- Output run directories ---
OUTPUT_A  = TRAIN_DIR / 'proposal_hybrid_resnet50_malds'
OUTPUT_B  = TRAIN_DIR / 'hybrid_resnet50_staged'
INFER_OUT = INFER_DIR / 'proposal_hybrid_resnet50_malds'

# --- Training hyperparameters ---
PHASE1_EPOCHS = 30   # frozen backbone
PHASE2_EPOCHS = 90   # full finetune
BATCH_SIZE    = 16   # T4 VRAM ~15GB, aman untuk image 320px
IMAGE_SIZE    = 320
NUM_WORKERS   = 4

print('=== PATH CONFIGURATION ===')
print(f'Image dir      : {IMAGE_DIR}')
print(f'Annotation     : {ANNOT_JSON}')
print(f'Scripts        : {SCRIPTS_DIR}')
print(f'Output         : {OUTPUT_ROOT}')
print(f'Device         : {DEVICE}')
print(f'Batch / ImgSz  : {BATCH_SIZE} / {IMAGE_SIZE}px')
print(f'Epochs P1+P2   : {PHASE1_EPOCHS} + {PHASE2_EPOCHS}')

In [ ]:
# Validasi semua path
errors = []

if IMAGE_DIR is None or not IMAGE_DIR.exists():
    errors.append(f'❌ Image dir tidak ada: {IMAGE_DIR}')
if ANNOT_JSON is None or not ANNOT_JSON.exists():
    errors.append(f'❌ Annotation path tidak ada: {ANNOT_JSON}')
if not SCRIPTS_DIR.exists():
    errors.append(f'❌ Scripts dir tidak ada: {SCRIPTS_DIR}')

if errors:
    raise RuntimeError('\n'.join(errors) + '\n\nPerbaiki path di cell konfigurasi di atas, lalu jalankan ulang.')

jpgs = []
for pattern in ['*.jpg', '*.JPG', '*.jpeg', '*.JPEG']:
    jpgs.extend(IMAGE_DIR.glob(pattern))
levels = {lvl: sum(1 for j in jpgs if j.name.startswith(lvl))
          for lvl in ['levle0', 'levle1', 'levle2', 'levle3']}

# Audit identitas v2, bukan sekadar keberadaan path.
import importlib.util
from urllib.request import urlopen
V2_AUDIT_SCRIPT = SCRIPTS_DIR / 'audit_acne04v2.py'
if not V2_AUDIT_SCRIPT.is_file():
    audit_url = 'https://raw.githubusercontent.com/Pandeigede2803/acne_segmentation_github/main/scripts/audit_acne04v2.py'
    with urlopen(audit_url, timeout=60) as response:
        V2_AUDIT_SCRIPT.write_bytes(response.read())
audit_spec = importlib.util.spec_from_file_location('acne04v2_audit', V2_AUDIT_SCRIPT)
v2_audit = importlib.util.module_from_spec(audit_spec)
audit_spec.loader.exec_module(v2_audit)
source_audit = v2_audit.audit_source(IMAGE_DIR, ANNOT_JSON)
print('✅ Sumber ACNE04-v2 terverifikasi dengan JSON resmi')
print(f"   File gambar di sumber (rekursif): {source_audit['source_image_files']}")
print(f"   Gambar v2 yang layak: {source_audit['eligible_images']}")
print(f"   File sumber di luar subset v2: {source_audit['excluded_source_files']} (tidak masuk manifest)")
print(f'   Device       : {DEVICE}')
print()
print('Distribusi subset v2 menurut awalan nama file (bukan hasil hitung ulang lesi):')
for lvl, count in sorted(source_audit['class_counts_from_filename'].items()):
    bar = '█' * (count // 20)
    print(f'  {lvl}: {count:4d}  {bar}')


---
## 9. Preview Sample Gambar per Severity

In [ ]:
all_jpgs = []
for pattern in ['*.jpg', '*.JPG', '*.jpeg', '*.JPEG']:
    all_jpgs.extend(IMAGE_DIR.glob(pattern))
v2_filenames = {row['file_name'] for row in v2_audit.load_official_annotations(ANNOT_JSON)['images']}
all_jpgs = sorted(p for p in all_jpgs if p.name in v2_filenames)

samples = {}
for img_path in all_jpgs:
    for lvl in ['levle0', 'levle1', 'levle2', 'levle3']:
        if img_path.name.startswith(lvl) and lvl not in samples:
            samples[lvl] = img_path

if not samples:
    raise RuntimeError('Tidak ada sample levle0-levle3 ditemukan. Cek nama file gambar dataset.')

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax in axes:
    ax.axis('off')
for ax, lvl in zip(axes, ['levle0', 'levle1', 'levle2', 'levle3']):
    path = samples.get(lvl)
    if path is None:
        ax.set_title(f'{lvl} tidak ada', fontsize=12)
        continue
    img = Image.open(path).resize((256, 256))
    ax.imshow(img)
    ax.set_title(lvl, fontsize=12)

plt.suptitle('Sample Gambar per Severity Level', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(str(OUTPUT_ROOT / 'sample_images.png'), dpi=100)
plt.show()


---
## 10. Generate Refined Mask (Preprocessing)

Script `refine_masks_color.py` akan:
- Mendeteksi piksel lesi menggunakan Lab relative redness + HSV
- Menghasilkan refined mask yang lebih natural dari circle geometris
- Menyimpan input training tanpa equalize warna atau sharpen secara default
- Menggunakan letterbox agar rasio aspek foto tetap terjaga
- Membuat `refined_manifest.csv` siap training

**Estimasi waktu: 5–15 menit untuk 1204 gambar di T4 GPU.** Audit ketat menolak manifest yang tidak cocok dengan 1.204 gambar dan jumlah lesi JSON resmi. Folder output baru tidak memakai cache preprocessing lain. `audit_acne04v2.py` harus tersedia di SCRIPTS_DIR (unggah manual atau publish ke GitHub). Jangan menghapus hasil lama; gunakan folder baru jika provenance tidak cocok.

In [ ]:
import ast
import subprocess
import sys
from urllib.request import urlopen

if not REFINE_SCRIPT.exists():
    raise FileNotFoundError(f'Script tidak ditemukan: {REFINE_SCRIPT}')
refine_help = subprocess.run([sys.executable, str(REFINE_SCRIPT), '--help'],
                             check=True, capture_output=True, text=True).stdout
if '--resize-mode' not in refine_help or '--preprocessing-mode' not in refine_help:
    refine_url = 'https://raw.githubusercontent.com/Pandeigede2803/acne_segmentation_github/main/scripts/refine_masks_color.py'
    print('Script refinement lama. Mengambil versi terbaru dari GitHub...')
    with urlopen(refine_url, timeout=60) as response:
        refine_source = response.read().decode('utf-8')
    ast.parse(refine_source)
    if '--resize-mode' not in refine_source or '--preprocessing-mode' not in refine_source:
        raise RuntimeError('Script refinement terbaru belum tersedia di branch main. Push perubahan script terlebih dahulu.')
    REFINE_SCRIPT.write_text(refine_source, encoding='utf-8')
    print('Script refinement berhasil diperbarui.')

if 'v2_audit' not in globals():
    raise RuntimeError('Jalankan validasi dataset Step 8 dahulu.')
v2_audit.audit_source(IMAGE_DIR, ANNOT_JSON)
V2_PREPROCESS_RECORD = MASK_DIR / 'acne04v2_preprocessing_provenance.json'
if REFINED_MANIFEST.exists():
    current_record = v2_audit.preprocessing_provenance(REFINED_MANIFEST, ANNOT_JSON, IMAGE_DIR, REFINE_SCRIPT)
    if not V2_PREPROCESS_RECORD.is_file() or json.loads(V2_PREPROCESS_RECORD.read_text()) != current_record:
        raise ValueError('Manifest/provenance tidak sesuai v2 resmi. Pilih MASK_DIR baru dan buat preprocessing baru; jangan hapus hasil lama.')
    df = pd.read_csv(REFINED_MANIFEST)
    print(f'ℹ️  Manifest sudah ada ({len(df)} baris) — skip generate.')
    print(df['split'].value_counts().to_string())
elif not REFINE_SCRIPT.exists():
    print('❌ Script tidak ditemukan:', REFINE_SCRIPT)
else:
    subprocess.run([sys.executable, str(REFINE_SCRIPT), '--image-dir', str(IMAGE_DIR),
                    '--annotations-json', str(ANNOT_JSON), '--output-dir', str(MASK_DIR),
                    '--resize-mode', 'letterbox', '--preprocessing-mode', 'none',
                    '--target-size', '320', '--max-images', '0'], check=True)

    if REFINED_MANIFEST.exists():
        df = pd.read_csv(REFINED_MANIFEST)
        print(f'\n✅ Manifest dibuat: {len(df)} baris')
        print(df['split'].value_counts().to_string())
        current_record = v2_audit.preprocessing_provenance(REFINED_MANIFEST, ANNOT_JSON, IMAGE_DIR, REFINE_SCRIPT)
        V2_PREPROCESS_RECORD.write_text(json.dumps(current_record, indent=2), encoding='utf-8')
    else:
        raise RuntimeError('Gagal generate manifest. Training tidak boleh dilanjutkan.')

In [ ]:
# Preview refined mask (3-panel: original | circle mask | refined mask)
viz_folder = MASK_DIR / 'visualizations'
viz_files  = sorted(viz_folder.glob('*.png'))[:8] if viz_folder.exists() else []

if viz_files:
    fig, axes = plt.subplots(2, 4, figsize=(20, 10))
    for ax, vf in zip(axes.flatten(), viz_files):
        img = Image.open(vf)
        ax.imshow(img)
        ax.set_title(vf.stem[:28], fontsize=7)
        ax.axis('off')
    plt.suptitle('Refined Mask — 3 panel: original | circle | refined', fontsize=12)
    plt.tight_layout()
    plt.savefig(str(OUTPUT_ROOT / 'refined_mask_preview.png'), dpi=80)
    plt.show()
else:
    print('Folder visualizations kosong atau belum ada.')

---
## 10a. Augmentasi Offline Foto-Mask 100% (wajib untuk fokus ini)
Jalankan setelah audit Step 8 dan preprocessing Step 10 menghasilkan manifest 1.204 citra v2 unik. Split sudah ditetapkan; hanya train ditambah pasangan sintetis sampai jumlah kelas train asli terbesar per kelas. Validation/test disalin tanpa perubahan. Jumlah baris setelah augmentasi bukan jumlah pasien/citra asli baru. Cache yang tidak memiliki provenance cocok ditolak. Gunakan output baru tanpa menghapus hasil lain.


In [ ]:
OFFLINE_AUGMENT = True  # Augmentasi offline aktif untuk training biasa.
AUGMENT_TARGET_RATIO = 1.0  # Target semua kelas train ke jumlah kelas terbesar.
AUGMENTED_DIR = MASK_DIR / 'offline_augmented_pairs_100pct'
AUGMENTED_MANIFEST = MASK_DIR / 'refined_manifest_offline_augmented_100pct.csv'
AUGMENT_SCRIPT = SCRIPTS_DIR / 'augment_train_pairs_offline.py'
if OFFLINE_AUGMENT:
    if not REFINED_MANIFEST.exists():
        raise FileNotFoundError('Jalankan Step 10 terlebih dahulu.')
    if 'v2_audit' not in globals():
        raise RuntimeError('Jalankan audit dataset Step 8 dahulu.')
    v2_audit.audit_manifest(REFINED_MANIFEST, ANNOT_JSON, IMAGE_DIR)
    from urllib.request import urlopen
    augment_url = 'https://raw.githubusercontent.com/Pandeigede2803/acne_segmentation_github/main/scripts/augment_train_pairs_offline.py'
    with urlopen(augment_url, timeout=60) as response:
        AUGMENT_SCRIPT.write_bytes(response.read())
    print('Augmentation script terbaru diunduh dari GitHub.')
    AUGMENT_RECORD = MASK_DIR / 'offline_aug100_provenance.json'
    augment_protocol = {'source_sha256': v2_audit.sha256(REFINED_MANIFEST),
                        'annotations_sha256': v2_audit.sha256(ANNOT_JSON),
                        'script_sha256': v2_audit.sha256(AUGMENT_SCRIPT),
                        'target_ratio': AUGMENT_TARGET_RATIO, 'seed': 42}
    if AUGMENTED_MANIFEST.exists():
        if not AUGMENT_RECORD.is_file() or json.loads(AUGMENT_RECORD.read_text()) != augment_protocol:
            raise ValueError('Cache augmentasi berbeda/tanpa provenance. Gunakan MASK_DIR baru; jangan pakai cache lama.')
    if not AUGMENTED_MANIFEST.exists():
        import subprocess, sys
        subprocess.run([sys.executable, str(AUGMENT_SCRIPT), '--manifest', str(REFINED_MANIFEST),
                        '--output-dir', str(AUGMENTED_DIR), '--output-manifest', str(AUGMENTED_MANIFEST),
                        '--target-ratio', str(AUGMENT_TARGET_RATIO), '--seed', '42'], check=True)
    augmented_audit = v2_audit.audit_augmented_manifest(REFINED_MANIFEST, AUGMENTED_MANIFEST)
    if not AUGMENT_RECORD.exists():
        AUGMENT_RECORD.write_text(json.dumps(augment_protocol, indent=2), encoding='utf-8')
    print('Dataset sumber: 1.204 citra v2 unik; pasangan train sintetis:', augmented_audit['synthetic_train_pairs'])
    df_aug = pd.read_csv(AUGMENTED_MANIFEST)
    print('Train rows:', int((df_aug.split == 'train').sum()))
    print(df_aug[df_aug.split == 'train'].groupby('label_name').size())
else:
    print('Offline augmentation nonaktif; training memakai manifest asli.')


---
## 11. Training Proposed + Augmentasi Offline 100%

### Catatan sebelum menjalankan cell training
- **Training dari awal:** gunakan `TRAINING_MODE = 'new'` dan `RUN_ID` baru yang belum dipakai, misalnya `RUN_ID = 'proposed_offline_aug100_balanced_run02'`. Edit baris `RUN_ID` di cell kode berikut. Mengulang cell dengan nama yang sama tidak membuat eksperimen baru.
- **Melanjutkan training terputus:** gunakan `TRAINING_MODE = 'resume'` dan `RUN_ID` yang sama, dengan konfigurasi serta sumber yang tidak berubah. Jangan memakai checkpoint dari eksperimen lain.
- Jika muncul pesan **folder run sudah berisi file**, jangan hapus hasil lama. Pilih nama run baru untuk mengulang dari awal, atau resume jika checkpoint dan provenance run tersebut tersedia.
- Manifest augmented dari Step 10a boleh digunakan kembali jika sumber dan protokolnya sama; manifest lama tidak berarti checkpoint training lama ikut dipakai. Jangan mengubah foto/mask saat resume.
- Catat `Output:` yang dicetak cell. Setelah runtime reconnect, pulihkan `RUN_ID`, path, dan konfigurasi yang sama sebelum melanjutkan.

Fokus cell ini hanya model lengkap **proposed + augmentasi offline 100%**. Jalankan Step 10a dahulu dengan `OFFLINE_AUGMENT=True` dan `AUGMENT_TARGET_RATIO=1.0`: setiap kelas train ditambah sampai jumlah kelas train terbesar, bukan semua data digandakan. Validation/test tetap asli. Model memakai Spatial Attention, dilated decoder, AMFM, MA-LDS, class weighting balanced, tanpa weighted sampler; augmentasi online trainer tetap aktif. Default 30 epoch freeze + 90 fine-tuning mengikuti konfigurasi pusat. Folder run baru tidak menimpa hasil lama. Pilih `resume` hanya untuk run ini dengan konfigurasi dan file sumber yang sama; jangan resume checkpoint eksperimen lain. Test dievaluasi setelah training, bukan untuk memilih konfigurasi. Step 18–23 tidak diperlukan untuk training ini dan tidak diubah.

In [ ]:
# Proposed lengkap + offline augmentation 100%; bukan baseline ablation.
import json
import hashlib
from pathlib import Path
import pandas as pd
import subprocess
import sys
import ast
from urllib.request import urlopen

EXPERIMENT_DATE = '2026-11-11'
CLASS_WEIGHTING = 'balanced'
TRAINING_MODE = 'new'  # Pilih 'new' untuk folder kosong atau 'resume' untuk melanjutkan folder RUN_ID yang sama.
RUN_ID = f'{EXPERIMENT_DATE}_acne04v2_official1204_proposed_offline_aug100_balanced_v4'
assert CLASS_WEIGHTING == 'balanced'
assert TRAINING_MODE in ('new', 'resume')
OUTPUT_A = TRAIN_DIR / RUN_ID
INFER_OUT = INFER_DIR / OUTPUT_A.name

if not globals().get('OFFLINE_AUGMENT', False) or globals().get('AUGMENT_TARGET_RATIO') != 1.0:
    raise RuntimeError('Jalankan Step 10a dengan OFFLINE_AUGMENT=True dan AUGMENT_TARGET_RATIO=1.0.')
if 'AUGMENTED_MANIFEST' not in globals() or not AUGMENTED_MANIFEST.is_file():
    raise FileNotFoundError('Manifest augmented belum ada. Jalankan Step 10a dahulu.')
if not TRAIN_SCRIPT_A.exists():
    raise FileNotFoundError(f'Script tidak ditemukan: {TRAIN_SCRIPT_A}')
if not REFINED_MANIFEST.exists():
    raise FileNotFoundError('refined_manifest.csv belum ada. Jalankan preprocessing dulu.')
if 'v2_audit' not in globals():
    raise RuntimeError('Jalankan audit dataset Step 8 dahulu.')
dataset_audit = v2_audit.audit_manifest(REFINED_MANIFEST, ANNOT_JSON, IMAGE_DIR)
v2_audit.audit_augmented_manifest(REFINED_MANIFEST, AUGMENTED_MANIFEST)
preprocessing_record = REFINED_MANIFEST.parent / 'acne04v2_preprocessing_provenance.json'
if not preprocessing_record.is_file() or json.loads(preprocessing_record.read_text()) != v2_audit.preprocessing_provenance(REFINED_MANIFEST, ANNOT_JSON, IMAGE_DIR, REFINE_SCRIPT):
    raise ValueError('Provenance preprocessing tidak sesuai. Jalankan Step 10 pada folder baru.')
original_df = pd.read_csv(REFINED_MANIFEST, dtype=str, keep_default_na=False)
augmented_df = pd.read_csv(AUGMENTED_MANIFEST, dtype=str, keep_default_na=False)
required = {'split', 'label_index', 'filename'}
if not required.issubset(original_df.columns) or set(original_df.columns) != set(augmented_df.columns):
    raise ValueError('Kolom manifest asli/augmented tidak sesuai.')
if not set(augmented_df['split']).issubset({'train', 'val', 'test'}):
    raise ValueError('Split manifest tidak dikenal.')
before = original_df[original_df.split == 'train'].groupby('label_index').size()
after = augmented_df[augmented_df.split == 'train'].groupby('label_index').size()
if set(before.index) != {'0', '1', '2', '3'} or set(after.index) != set(before.index) or not after.eq(before.max()).all():
    raise ValueError('Train augmented harus memiliki empat kelas, masing-masing sebanyak kelas train asli terbesar.')
def canonical_rows(frame):
    return sorted(map(tuple, frame[sorted(original_df.columns)].to_numpy().tolist()))
for split in ('val', 'test'):
    if original_df[original_df.split == split].empty or canonical_rows(original_df[original_df.split == split]) != canonical_rows(augmented_df[augmented_df.split == split]):
        raise ValueError(f'Split {split} berubah atau kosong; validation/test harus tetap asli.')
from collections import Counter
if Counter(canonical_rows(original_df)) - Counter(canonical_rows(augmented_df)):
    raise ValueError('Sebagian baris asli hilang/berubah pada manifest augmented.')
print(pd.DataFrame({'train_asli': before, 'train_aug100': after}))
help_result = subprocess.run([sys.executable, str(TRAIN_SCRIPT_A), '--help'],
                             check=True, capture_output=True, text=True)
if ('--label-distribution' not in help_result.stdout or '--class-weighting' not in help_result.stdout or '--variant' not in help_result.stdout or
        '--skip-test-eval' not in help_result.stdout or 'phase1_optimizer_state_dict' not in TRAIN_SCRIPT_A.read_text()):
    script_url = 'https://raw.githubusercontent.com/Pandeigede2803/acne_segmentation_github/main/scripts/train_proposal_hybrid_resnet50_malds.py'
    print('Script perlu diperbarui. Mengambil script training dari GitHub...')
    with urlopen(script_url, timeout=60) as response:
        source = response.read().decode('utf-8')
    ast.parse(source)
    if any(flag not in source for flag in ['--label-distribution', '--class-weighting', '--variant', '--skip-test-eval', 'phase1_optimizer_state_dict']):
        raise RuntimeError('Script training terbaru belum tersedia di branch main. Push script terbaru terlebih dahulu.')
    TRAIN_SCRIPT_A.write_text(source, encoding='utf-8')
    help_result = subprocess.run([sys.executable, str(TRAIN_SCRIPT_A), '--help'],
                                 check=True, capture_output=True, text=True)
    assert all(flag in help_result.stdout for flag in ('--variant', '--label-distribution', '--skip-test-eval')), 'Script belum mendukung konfigurasi training.'
    print('Script training berhasil diperbarui.')
CHECKPOINT_EXISTS = (OUTPUT_A / 'last_checkpoint.pt').exists()
if TRAINING_MODE == 'new' and OUTPUT_A.exists() and any(OUTPUT_A.iterdir()):
    raise FileExistsError(f'Folder run sudah berisi file: {OUTPUT_A}. Ganti RUN_ID atau gunakan resume untuk run yang sama.')
if TRAINING_MODE == 'resume' and not CHECKPOINT_EXISTS:
    raise FileNotFoundError(f'Tidak ada checkpoint untuk resume di {OUTPUT_A}. Pilih TRAINING_MODE=new.')
RESUME = TRAINING_MODE == 'resume'

cmd = [
    sys.executable, '-u', str(TRAIN_SCRIPT_A),
    '--manifest', str(AUGMENTED_MANIFEST),
    '--output-dir', str(OUTPUT_A),
    '--variant', 'proposed', '--label-distribution', 'malds',
    '--class-weighting', CLASS_WEIGHTING,
    '--image-size', str(IMAGE_SIZE), '--batch-size', str(BATCH_SIZE),
    '--phase1-epochs', str(PHASE1_EPOCHS), '--phase2-epochs', str(PHASE2_EPOCHS),
    '--device', DEVICE, '--pretrained',
    '--num-workers', str(min(NUM_WORKERS, 2)), '--seed', '42',
]
# Catat sumber dan konfigurasi sebelum training; resume menolak perubahan.
provenance_path = OUTPUT_A / 'augmentation_training_provenance.json'
provenance = {
    'command': cmd, 'target_ratio': 1.0, 'weighted_sampler': False,
    'dataset_audit': dataset_audit,
    'train_counts_before': {str(k): int(v) for k, v in before.items()},
    'train_counts_after': {str(k): int(v) for k, v in after.items()},
    'sha256': {str(p): hashlib.sha256(p.read_bytes()).hexdigest()
               for p in (REFINED_MANIFEST, AUGMENTED_MANIFEST, TRAIN_SCRIPT_A, ANNOT_JSON, REFINE_SCRIPT)},
}
if RESUME:
    if not provenance_path.is_file() or json.loads(provenance_path.read_text()) != provenance:
        raise ValueError('Provenance resume berbeda/tidak tersedia. Pulihkan konfigurasi asli atau buat RUN_ID baru; jangan timpa hasil lama.')
else:
    OUTPUT_A.mkdir(parents=True, exist_ok=True)
    provenance_path.write_text(json.dumps(provenance, indent=2), encoding='utf-8')
if RESUME:
    cmd.append('--resume')
    print('Resume dipilih. Melanjutkan dari checkpoint pada RUN_ID:', RUN_ID)
else:
    print('Memulai training baru.')
print('Model: proposed lengkap | MA-LDS | offline augmentation 100% | weighted sampler: False')
print('Epoch freeze / fine-tuning:', PHASE1_EPOCHS, '/', PHASE2_EPOCHS)
print('Class weighting:', CLASS_WEIGHTING)
print('Output:', OUTPUT_A)
process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                           text=True, bufsize=1)
try:
    for line in process.stdout:
        print(line, end='', flush=True)
    returncode = process.wait()
    if returncode != 0:
        raise subprocess.CalledProcessError(returncode, cmd)
finally:
    if process.poll() is None:
        process.terminate()
        try:
            process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()
    process.stdout.close()

---
## 12. Monitor Training (jalankan sambil training berjalan)

In [ ]:
# Ganti ke OUTPUT_B / 'metrics.json' jika menjalankan Option B
METRICS_FILE = OUTPUT_A / 'metrics.json'

def plot_training_curves(metrics_path: Path):
    if not metrics_path.exists():
        print('metrics.json belum ada — training mungkin belum mulai.')
        return

    with open(metrics_path) as f:
        data = json.load(f)

    history = data.get('history', [])
    if not history:
        print('History kosong.')
        return

    epochs     = [h['epoch'] for h in history]
    train_loss = [h.get('train_loss', 0) for h in history]
    val_loss   = [h.get('val_loss', 0) for h in history]
    train_dice = [h.get('train_dice', 0) for h in history]
    val_dice   = [h.get('val_dice', 0) for h in history]
    train_acc  = [h.get('train_acc', 0) for h in history]
    val_acc    = [h.get('val_acc', 0) for h in history]
    train_kappa= [h.get('train_kappa', 0) for h in history]
    val_kappa  = [h.get('val_kappa', 0) for h in history]

    fig, axes = plt.subplots(2, 2, figsize=(14, 8))

    axes[0, 0].plot(epochs, train_loss, label='Train', color='steelblue')
    axes[0, 0].plot(epochs, val_loss,   label='Val',   color='orange')
    axes[0, 0].set_title('Loss'); axes[0, 0].legend(); axes[0, 0].grid(alpha=0.3)

    axes[0, 1].plot(epochs, train_dice, label='Train', color='green')
    axes[0, 1].plot(epochs, val_dice,   label='Val',   color='red')
    axes[0, 1].set_title('Dice Score'); axes[0, 1].legend(); axes[0, 1].grid(alpha=0.3)

    axes[1, 0].plot(epochs, train_acc, label='Train', color='steelblue')
    axes[1, 0].plot(epochs, val_acc,   label='Val',   color='orange')
    axes[1, 0].set_title('Accuracy'); axes[1, 0].legend(); axes[1, 0].grid(alpha=0.3)

    axes[1, 1].plot(epochs, train_kappa, label='Train', color='purple')
    axes[1, 1].plot(epochs, val_kappa,   label='Val',   color='darkred')
    axes[1, 1].set_title("Cohen's Kappa"); axes[1, 1].legend(); axes[1, 1].grid(alpha=0.3)

    last = history[-1]
    fig.suptitle(
        f"Epoch {last['epoch']} | Phase: {last.get('phase', '?')} | "
        f"Val Dice: {last.get('val_dice', 0):.4f} | Val Acc: {last.get('val_acc', 0):.4f}",
        fontsize=12
    )
    plt.tight_layout()
    plt.savefig(str(OUTPUT_ROOT / 'training_curves.png'), dpi=100, bbox_inches='tight')
    plt.show()
    print(f'Epoch {last["epoch"]} | Val Dice: {last.get("val_dice",0):.4f} | Val Acc: {last.get("val_acc",0):.4f}')

plot_training_curves(METRICS_FILE)

---
## 13. Test Metrics

In [ ]:
def print_test_metrics(metrics_path: Path):
    if not metrics_path.exists():
        print('❌ metrics.json tidak ditemukan:', metrics_path)
        return

    with open(metrics_path) as f:
        data = json.load(f)

    tm  = data.get('test_metrics', {})
    cfg = data.get('config', {})

    print('=' * 45)
    print('       TEST METRICS — HASIL AKHIR')
    print('=' * 45)
    print(f"  Accuracy      : {tm.get('acc', 0):.4f}")
    print(f"  Cohen Kappa   : {tm.get('kappa', 0):.4f}")
    print(f"  Dice Score    : {tm.get('dice', 0):.4f}")
    print(f"  IoU           : {tm.get('iou', 0):.4f}")
    print('-' * 45)
    print(f"  Total Loss    : {tm.get('loss', 0):.4f}")
    print(f"  Seg Loss      : {tm.get('seg_loss', 0):.4f}")
    print(f"  Cls Loss      : {tm.get('cls_loss', 0):.4f}")
    print('=' * 45)
    print(f"  Best Val Loss : {data.get('best_val_loss', 0):.4f}")
    print(f"  Epochs (P1+P2): {cfg.get('phase1_epochs', '?')} + {cfg.get('phase2_epochs', '?')}")
    print(f"  Batch size    : {cfg.get('batch_size', '?')}")
    print(f"  Image size    : {cfg.get('image_size', '?')}")

print_test_metrics(OUTPUT_A / 'metrics.json')

---
## 14. Inference — Prediksi Test Set

In [ ]:
import ast
import subprocess
import sys
from urllib.request import urlopen

if not INFER_SCRIPT.exists():
    raise FileNotFoundError(f'Script inferensi tidak ditemukan: {INFER_SCRIPT}')
infer_help = subprocess.run([sys.executable, str(INFER_SCRIPT), '--help'],
                            check=True, capture_output=True, text=True).stdout
if '--input-normalization' not in infer_help:
    infer_url = 'https://raw.githubusercontent.com/Pandeigede2803/acne_segmentation_github/main/scripts/infer_testing_images.py'
    print('Script inferensi lama. Mengambil versi terbaru dari GitHub...')
    with urlopen(infer_url, timeout=60) as response:
        infer_source = response.read().decode('utf-8')
    ast.parse(infer_source)
    if '--input-normalization' not in infer_source:
        raise RuntimeError('Script inferensi terbaru belum tersedia di branch main. Push perubahan script terlebih dahulu.')
    INFER_SCRIPT.write_text(infer_source, encoding='utf-8')
    print('Script inferensi berhasil diperbarui.')

CHECKPOINT = OUTPUT_A / 'best_model.pt'

if not INFER_SCRIPT.exists():
    print('❌ Script tidak ditemukan:', INFER_SCRIPT)
elif not CHECKPOINT.exists():
    print('❌ Checkpoint tidak ditemukan:', CHECKPOINT)
    print('   Training harus selesai dulu.')
elif not REFINED_MANIFEST.exists():
    print('❌ refined_manifest.csv tidak ada.')
else:
    !python "{INFER_SCRIPT}" \
      --manifest    "{REFINED_MANIFEST}" \
      --checkpoint  "{CHECKPOINT}" \
      --output-dir  "{INFER_OUT}" \
      --split test \
      --image-size  320 \
      --threshold   0.5 \
      --device      {DEVICE} \
      --model-type  proposal \
      --input-normalization auto

---
## 15. Visualisasi Hasil Segmentasi

In [ ]:
# Preview predicted mask vs ground truth
pred_folder = INFER_OUT / 'overlays'
if not pred_folder.exists():
    pred_folder = INFER_OUT

pred_files = sorted(pred_folder.glob('*.png'))[:8]

if pred_files:
    fig, axes = plt.subplots(2, 4, figsize=(20, 10))
    for ax, pf in zip(axes.flatten(), pred_files):
        img = Image.open(pf)
        ax.imshow(img)
        ax.set_title(pf.stem[:25], fontsize=8)
        ax.axis('off')
    plt.suptitle('Predicted Segmentation Mask — Test Set', fontsize=14)
    plt.tight_layout()
    plt.savefig(str(OUTPUT_ROOT / 'predicted_masks_preview.png'), dpi=100)
    plt.show()
else:
    print('Folder output inferensi kosong atau belum ada.')

In [ ]:
# Confusion matrix
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

infer_csv = INFER_OUT / 'testing_predictions.csv'

if not infer_csv.exists():
    print('testing_predictions.csv belum ada. Jalankan inferensi dulu.')
else:
    df_pred = pd.read_csv(infer_csv)
    print(f'Total prediksi: {len(df_pred)}')
    print(df_pred.head())

    if 'true_label_name' in df_pred.columns and 'pred_label_name' in df_pred.columns:
        labels = ['levle0', 'levle1', 'levle2', 'levle3']
        cm = confusion_matrix(df_pred['true_label_name'], df_pred['pred_label_name'], labels=labels)

        fig, ax = plt.subplots(figsize=(7, 6))
        disp = ConfusionMatrixDisplay(cm, display_labels=labels)
        disp.plot(ax=ax, colorbar=False, cmap='Blues')
        ax.set_title('Confusion Matrix — Test Set', fontsize=13)
        plt.tight_layout()
        plt.savefig(str(OUTPUT_ROOT / 'confusion_matrix.png'), dpi=100)
        plt.show()
    else:
        print('Kolom true_label_name/pred_label_name tidak ditemukan di testing_predictions.csv')


---
## 16. Rekap Semua Eksperimen

In [ ]:
result_files = list(TRAIN_DIR.rglob('experiment_results.csv'))
metrics_files = list(TRAIN_DIR.rglob('metrics.json'))

# Baca dari experiment_results.csv jika ada
if result_files:
    dfs = [pd.read_csv(f) for f in result_files]
    df_all = pd.concat(dfs, ignore_index=True)
    cols_show = ['run_name', 'encoder', 'epochs', 'test_acc', 'test_kappa', 'test_dice', 'test_iou']
    cols_exist = [c for c in cols_show if c in df_all.columns]
    print('=== REKAP SEMUA RUN ===')
    print(df_all[cols_exist].to_string(index=False))

# Fallback: baca dari masing-masing metrics.json
elif metrics_files:
    print('=== REKAP DARI metrics.json ===')
    rows = []
    for mf in metrics_files:
        with open(mf) as f:
            d = json.load(f)
        tm = d.get('test_metrics', {})
        rows.append({
            'run'  : mf.parent.name,
            'dice' : round(tm.get('dice', 0), 4),
            'iou'  : round(tm.get('iou', 0), 4),
            'acc'  : round(tm.get('acc', 0), 4),
            'kappa': round(tm.get('kappa', 0), 4),
        })
    print(pd.DataFrame(rows).to_string(index=False))

else:
    print('Belum ada hasil eksperimen. Training harus selesai dulu.')

---
## 17. Backup Checkpoint ke Drive

Output training sudah langsung disimpan ke Drive. Cell ini hanya untuk memastikan tidak ada checkpoint yang tertinggal di `/content/`.

In [ ]:
# Copy semua checkpoint dari /content/ ke Drive jika belum ada
checkpoints = list(Path('/content').rglob('best_model.pt'))
copied = 0

for ckpt in checkpoints:
    rel = ckpt.relative_to('/content')
    target = OUTPUT_ROOT / rel
    target.parent.mkdir(parents=True, exist_ok=True)
    if not target.exists():
        shutil.copy2(ckpt, target)
        print(f'✅ Disalin: {ckpt} → {target}')
        copied += 1
    else:
        print(f'ℹ️  Sudah ada: {target}')

if not checkpoints:
    print('Tidak ada checkpoint di /content/')
else:
    print(f'\n{copied} file dicopy. Semua output tersimpan di: {OUTPUT_ROOT}')

---
## 18. Alur Opsional Sesuai Flowchart — Audit Filter Foto

Bagian ini ditambahkan tanpa menghapus alur lama. Untuk menjalankan jalur flowchart dari runtime baru, jalankan setup dan preprocessing sampai Step 10, lalu lewati training legacy Steps 11–17 dan lanjutkan ke Step 18. Secara default semua foto dimasukkan (`include=yes`) agar eksperimen bisa dilanjutkan tanpa audit manual. Jika ingin mengecualikan foto, ubah `include` menjadi `no` dan isi alasannya. Pemeriksaan visual tetap bisa dilakukan lewat contact sheet.

Jalankan cell daftar review untuk membuat contact sheet dan CSV. Cell penerapan filter akan mengubah status `pending` yang tersisa menjadi `yes` secara otomatis.

In [ ]:
import ast
import subprocess
import sys
from urllib.request import urlopen

train_help = subprocess.run([sys.executable, str(TRAIN_SCRIPT_A), '--help'], check=True, capture_output=True, text=True).stdout
if '--variant' not in train_help or '--skip-test-eval' not in train_help or 'phase1_optimizer_state_dict' not in TRAIN_SCRIPT_A.read_text():
    train_url = 'https://raw.githubusercontent.com/Pandeigede2803/acne_segmentation_github/main/scripts/train_proposal_hybrid_resnet50_malds.py'
    print('Script training belum mendukung eksperimen flowchart. Mengambil versi terbaru dari GitHub...')
    with urlopen(train_url, timeout=60) as response:
        train_source = response.read().decode('utf-8')
    ast.parse(train_source)
    if any(flag not in train_source for flag in ['--variant', '--skip-test-eval', 'phase1_optimizer_state_dict']):
        raise RuntimeError('Script eksperimen terbaru belum ada di branch main. Push script training terbaru dahulu.')
    TRAIN_SCRIPT_A.write_text(train_source, encoding='utf-8')
    print('Script eksperimen berhasil diperbarui.')

FLOWCHART_DIR = OUTPUT_ROOT / 'flowchart_experiments_2026-10-24'
FLOWCHART_DIR.mkdir(parents=True, exist_ok=True)
FILTER_REVIEW_CSV = FLOWCHART_DIR / 'photo_filter_review.csv'
FILTERED_MANIFEST = FLOWCHART_DIR / 'refined_manifest_filtered.csv'

if not REFINED_MANIFEST.exists():
    raise FileNotFoundError(f'Manifest belum ada: {REFINED_MANIFEST}')
df_manifest = pd.read_csv(REFINED_MANIFEST)
if not FILTER_REVIEW_CSV.exists():
    review_cols = ['filename', 'label_name', 'split', 'image_path', 'n_circles', 'fallback', 'include', 'exclusion_reason']
    review = df_manifest[review_cols[:6]].copy()
    review['include'] = 'yes'
    review['exclusion_reason'] = ''
    review.to_csv(FILTER_REVIEW_CSV, index=False)
    print(f'CSV review dibuat: {FILTER_REVIEW_CSV}')
    print('Semua foto otomatis diset include=yes. Ubah menjadi no hanya untuk foto yang ingin dikeluarkan.')
else:
    print(f'CSV review sudah ada: {FILTER_REVIEW_CSV}')
print('Jumlah foto:', len(df_manifest))
print(df_manifest.groupby(['split', 'label_name']).size().unstack(fill_value=0))

# Buat contact sheet agar foto bisa ditinjau visual sebelum keep/exclude.
from PIL import Image, ImageDraw
from IPython.display import display
REVIEW_SHEET_DIR = FLOWCHART_DIR / 'photo_review_sheets'
REVIEW_SHEET_DIR.mkdir(parents=True, exist_ok=True)
TILE_W, TILE_H, COLS, ROWS_PER_SHEET = 220, 250, 5, 6
SHEETS_PER_RUN = 5  # 5 lembar = maksimal 150 foto per eksekusi; jalankan cell lagi untuk lanjut.
SHEET_SIZE = COLS * TILE_W, ROWS_PER_SHEET * TILE_H
sheet_starts = list(range(0, len(df_manifest), COLS * ROWS_PER_SHEET))
completed = [start for start in sheet_starts if (REVIEW_SHEET_DIR / f"sheet_{start // (COLS * ROWS_PER_SHEET) + 1:03d}.jpg").exists()]
pending_starts = [start for start in sheet_starts if start not in completed]
run_starts = pending_starts[:SHEETS_PER_RUN]
print(f"Contact sheet selesai: {len(completed)}/{len(sheet_starts)}. Batch ini membuat {len(run_starts)} lembar.", flush=True)

for batch_pos, start in enumerate(run_starts, start=1):
    sheet_idx = start // (COLS * ROWS_PER_SHEET) + 1
    sheet_rows = df_manifest.iloc[start:start + COLS * ROWS_PER_SHEET]
    sheet = Image.new('RGB', SHEET_SIZE, 'white')
    draw = ImageDraw.Draw(sheet)
    for pos, (_, row) in enumerate(sheet_rows.iterrows()):
        image_path = Path(str(row.get('image_path', '')))
        if not image_path.exists() and row.get('preprocessed_path'):
            image_path = Path(str(row['preprocessed_path']))
        if not image_path.exists():
            continue
        with Image.open(image_path) as source_image:
            image = source_image.convert('RGB')
        image.thumbnail((TILE_W - 10, TILE_H - 42), Image.Resampling.LANCZOS)
        x = (pos % COLS) * TILE_W
        y = (pos // COLS) * TILE_H
        sheet.paste(image, (x + (TILE_W - image.width) // 2, y + 4))
        mask_value = row.get('refined_mask') or row.get('mask_path')
        mask_path = Path(str(mask_value)) if mask_value else Path('')
        if mask_value and mask_path.exists():
            with Image.open(mask_path) as source_mask:
                mask_img = source_mask.convert('L').resize(image.size, Image.Resampling.NEAREST)
            overlay = Image.new('RGBA', image.size, (255, 40, 40, 0))
            overlay.putalpha(mask_img.point(lambda value: 95 if value > 0 else 0))
            sheet.paste(overlay, (x + (TILE_W - image.width) // 2, y + 4), overlay)
        draw.text((x + 4, y + TILE_H - 34), f"{row['filename']} | {row['label_name']}", fill='black')
        draw.text((x + 4, y + TILE_H - 18), f"{row['split']} | fallback={row.get('fallback', '')}", fill='black')
    out = REVIEW_SHEET_DIR / f'sheet_{sheet_idx:03d}.jpg'
    sheet.save(out, quality=90)
    print(f"Batch {batch_pos}/{len(run_starts)}: sheet {sheet_idx}/{len(sheet_starts)} selesai ({min(start + COLS * ROWS_PER_SHEET, len(df_manifest))}/{len(df_manifest)} foto).", flush=True)

remaining = len(pending_starts) - len(run_starts)
print(f"Contact sheets tersimpan: {REVIEW_SHEET_DIR}. Batch ini selesai; tersisa {remaining} lembar.", flush=True)
if run_starts:
    display(Image.open(REVIEW_SHEET_DIR / f"sheet_{run_starts[0] // (COLS * ROWS_PER_SHEET) + 1:03d}.jpg"))
elif sheet_starts:
    print('Semua lembar sudah dibuat; menampilkan sheet pertama.')
    display(Image.open(REVIEW_SHEET_DIR / 'sheet_001.jpg'))
if remaining:
    print(f"Jalankan cell ini lagi untuk membuat batch berikutnya ({min(remaining, SHEETS_PER_RUN)} lembar).")
else:
    print('Semua contact sheet sudah selesai.')

In [ ]:
review = pd.read_csv(FILTER_REVIEW_CSV).fillna({'include': '', 'exclusion_reason': ''})
review['include'] = review['include'].astype(str).str.strip().str.lower()
# Default praktis: foto yang masih pending/kosong dianggap disertakan.
pending_mask = ~review['include'].isin(['yes', 'no'])
if pending_mask.any():
    pending_count = int(pending_mask.sum())
    review.loc[pending_mask, 'include'] = 'yes'
    review.to_csv(FILTER_REVIEW_CSV, index=False)
    print(f'{pending_count} foto berstatus pending otomatis diubah menjadi include=yes.')
if review['filename'].duplicated().any():
    raise ValueError('CSV review berisi filename duplikat.')
expected_names = set(df_manifest['filename'].astype(str))
review_names = set(review['filename'].astype(str))
if expected_names != review_names:
    raise ValueError(f'Daftar review berbeda dari manifest: hilang={len(expected_names - review_names)}, tambahan={len(review_names - expected_names)}')
missing_reason = review.loc[(review['include'] == 'no') & (review['exclusion_reason'].astype(str).str.strip() == ''), 'filename'].tolist()
if missing_reason:
    raise ValueError(f'Isi alasan exclusion untuk {len(missing_reason)} foto yang ditolak.')
pending = review.loc[~review['include'].isin(['yes', 'no']), 'filename'].tolist()
if pending:
    raise ValueError(f'Masih ada {len(pending)} foto yang belum direview. Isi include=yes/no dulu.')
keep_names = set(review.loc[review['include'] == 'yes', 'filename'])
filtered = df_manifest[df_manifest['filename'].isin(keep_names)].copy()
counts = filtered.groupby(['split', 'label_name']).size().unstack(fill_value=0).reindex(columns=['levle0', 'levle1', 'levle2', 'levle3'], fill_value=0)
print('Jumlah setelah review:', len(filtered))
print(counts)
if 'train' not in counts.index or (counts.loc['train'] == 0).any():
    raise ValueError('Filter menghilangkan setidaknya satu kelas dari split train; periksa keputusan review.')
if 'val' not in counts.index or (counts.loc['val'] == 0).any():
    raise ValueError('Filter menghilangkan setidaknya satu kelas dari split val; periksa keputusan review.')
if 'test' not in counts.index or (counts.loc['test'] == 0).any():
    raise ValueError('Filter menghilangkan setidaknya satu kelas dari split test; periksa keputusan review.')
filtered.to_csv(FILTERED_MANIFEST, index=False)
print(f'Manifest hasil filter tersimpan: {FILTERED_MANIFEST}')

---
## 19. 5-Fold Cross-Validation

Fold dibuat dari split `train` saja; split `val` dan `test` asli tidak dipakai di dalam fold. Test evaluation dilewati pada tiap fold agar test set tidak dipakai memilih model. Setiap fold memerlukan training penuh 30+90 epoch dan dapat memakan waktu lama. Cell pertama hanya menyiapkan manifest; cell kedua menjalankan training setelah `RUN_CV = True`.

In [ ]:
import random

if not FILTERED_MANIFEST.exists():
    raise FileNotFoundError('Jalankan audit foto dan buat refined_manifest_filtered.csv terlebih dahulu.')
cv_df = pd.read_csv(FILTERED_MANIFEST).fillna('')
train_pool = cv_df[cv_df['split'] == 'train'].to_dict('records')
fold_buckets = {i: [] for i in range(5)}
rng = random.Random(42)
for label in sorted({int(row['label_index']) for row in train_pool}):
    group = [row for row in train_pool if int(row['label_index']) == label]
    rng.shuffle(group)
    if len(group) < 5:
        raise ValueError(f'Kelas {label} hanya punya {len(group)} foto train; tidak cukup untuk 5 fold.')
    for i, row in enumerate(group):
        fold_buckets[i % 5].append(row)
CV_DIR = FLOWCHART_DIR / 'cross_validation'
CV_DIR.mkdir(parents=True, exist_ok=True)
CV_MANIFESTS = []
for fold in range(5):
    val_names = {row['filename'] for row in fold_buckets[fold]}
    fold_rows = []
    for row in train_pool:
        item = dict(row)
        item['split'] = 'val' if item['filename'] in val_names else 'train'
        fold_rows.append(item)
    path = CV_DIR / f'fold_{fold + 1}_manifest.csv'
    pd.DataFrame(fold_rows).to_csv(path, index=False)
    CV_MANIFESTS.append(path)
    print(f'Fold {fold + 1}: train={(len(fold_rows) - len(val_names))}, val={len(val_names)} -> {path}')

In [ ]:
RUN_CV = False  # Ubah ke True setelah manifest fold siap; 5 fold x 120 epoch memerlukan GPU lama.
if RUN_CV:
    import re
    import subprocess
    import sys
    import time
    from datetime import timedelta

    cv_runs = []
    cv_started = time.monotonic()
    total_epochs = PHASE1_EPOCHS + PHASE2_EPOCHS
    for i, fold_manifest in enumerate(CV_MANIFESTS, start=1):
        run_dir = CV_DIR / f'run_fold_{i}'
        cmd = [sys.executable, '-u', str(TRAIN_SCRIPT_A), '--manifest', str(fold_manifest),
               '--output-dir', str(run_dir), '--variant', 'proposed',
               '--class-weighting', 'balanced', '--weighted-sampler',
               '--image-size', str(IMAGE_SIZE), '--batch-size', str(BATCH_SIZE),
               '--phase1-epochs', str(PHASE1_EPOCHS), '--phase2-epochs', str(PHASE2_EPOCHS),
               '--device', DEVICE, '--pretrained', '--num-workers', str(min(NUM_WORKERS, 2)),
               '--seed', '42', '--skip-test-eval']
        print(f'\n===== CV fold {i}/5 dimulai: {fold_manifest.name} =====', flush=True)
        fold_started = time.monotonic()
        epoch_started = None
        epoch_durations = []
        last_epoch = 0
        with subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                              text=True, bufsize=1) as process:
            for line in process.stdout:
                print(line, end='', flush=True)
                if re.match(r'^📘 Epoch \d+/', line):
                    epoch_started = time.monotonic()
                match = re.match(r'^Epoch (\d+) \| train_loss=', line)
                if match:
                    last_epoch = int(match.group(1))
                    if epoch_started is not None:
                        epoch_durations.append(time.monotonic() - epoch_started)
                    if epoch_durations:
                        avg_epoch = sum(epoch_durations) / len(epoch_durations)
                        epochs_left_this_fold = max(0, total_epochs - last_epoch)
                        folds_after_this = len(CV_MANIFESTS) - i
                        eta_seconds = (epochs_left_this_fold + folds_after_this * total_epochs) * avg_epoch
                        print(f'⏱️ Progress CV: fold {i}/5, epoch {last_epoch}/{total_epochs}; '
                              f'rata-rata {avg_epoch / 60:.1f} menit/epoch; '
                              f'estimasi sisa CV sekitar {timedelta(seconds=int(eta_seconds))}.', flush=True)
            return_code = process.wait()
        if return_code != 0:
            raise subprocess.CalledProcessError(return_code, cmd)
        print(f'✅ Fold {i}/5 selesai dalam {timedelta(seconds=int(time.monotonic() - fold_started))}.', flush=True)
        summary = json.loads((run_dir / 'metrics.json').read_text())
        best = min(summary['history'], key=lambda row: row['val_loss'])
        cv_runs.append({'fold': i, 'best_epoch': best['epoch'], 'val_accuracy': best['val_acc'],
                        'val_kappa': best['val_kappa'], 'val_dice': best['val_dice'],
                        'val_iou': best['val_iou'], 'best_val_loss': summary['best_val_loss']})
        print(f'📊 Fold {i}/5 tercatat. Total waktu berjalan: '
              f'{timedelta(seconds=int(time.monotonic() - cv_started))}.', flush=True)
    cv_results = pd.DataFrame(cv_runs)
    cv_results.to_csv(CV_DIR / 'cross_validation_results.csv', index=False)
    display(cv_results)
    display(cv_results.mean(numeric_only=True).to_frame('mean').T)
else:
    print('CV belum dijalankan. Ubah RUN_CV=True untuk memulai training lima fold.')


---
## 20. Ablation Study — Augmentasi Offline 100%

### Catatan sebelum menjalankan cell ablation
- Default `RUN_ABLATIONS = False`. Ubah menjadi `True` hanya ketika siap menjalankan/melanjutkan tujuh training.
- **Melanjutkan ablation terputus:** gunakan `OUTPUT_A`, konfigurasi, dan folder `ABLATION_DIR` yang sama. Varian yang sudah memiliki `metrics.json` dilewati; yang memiliki checkpoint dilanjutkan. Pesan **sudah selesai; memakai metrics.json yang tersimpan** berarti hasil lama dibaca, bukan training baru.
- **Mengulang semua ablation dari awal pada sumber Step 11 yang sama:** ganti suffix folder pada baris `ABLATION_DIR`, misalnya dari `_ablation_aug100_v1` menjadi `_ablation_aug100_v2`. Jangan hapus atau timpa folder lama.
- Jika Step 11 memakai `RUN_ID` baru, jalankan Step 11 tersebut dahulu; folder ablation mengikuti `OUTPUT_A.name`. Pastikan `OUTPUT_A` menunjuk run augmented yang dimaksud, bukan run lama.
- Setelah reconnect, pulihkan variabel/path Step 10a dan Step 11. Jangan jalankan training Step 11 ulang hanya untuk memulihkan variabel; definisikan kembali variabel yang diperlukan dengan nilai persis sama.
- Jika muncul ketidaksesuaian provenance/hash, jangan melewati pemeriksaan atau menghapus provenance. Pulihkan sumber/config semula, atau siapkan run Step 11 dan folder ablation baru untuk protokol yang berubah.

Jalankan Step 10a dan Step 11 augmented dahulu (setelah reconnect, pulihkan variabel yang sama). Tujuh training: enam varian + proposed; manifest augmented yang sama, hanya train/validation. Tanpa weighted sampler; balanced class weighting; augmentasi online tetap aktif. Seed/epoch/batch diverifikasi terhadap catatan Step 11. Test tidak dievaluasi. Folder baru melindungi hasil lama. Resume mensyaratkan konfigurasi dan hash sumber sama. Jangan mengubah foto/mask; isi foto/mask belum di-hash. Default training OFF. Checkpoint dipilih dengan validation loss minimum per run; loss antar-objektif tidak otomatis setara. Baseline tetap multi-task, bukan single-task murni. Step 18/19/21 tidak diperlukan.

In [ ]:
import json
import hashlib
from pathlib import Path
import pandas as pd
ABLATION_VARIANTS = ['baseline', 'no_attention', 'no_dilation', 'no_amfm', 'lds', 'no_fusion', 'proposed']
RUN_ABLATIONS = False  # True: tujuh training baru/resume; False: tidak menulis hasil atau melatih.
if not globals().get('OFFLINE_AUGMENT', False) or globals().get('AUGMENT_TARGET_RATIO') != 1.0:
    raise RuntimeError('Jalankan Step 10a dengan augmentasi offline 100%.')
if 'AUGMENTED_MANIFEST' not in globals() or not AUGMENTED_MANIFEST.is_file():
    raise FileNotFoundError('Manifest augmented belum tersedia.')
if 'OUTPUT_A' not in globals():
    raise RuntimeError('Pulihkan konfigurasi Step 11 augmented terlebih dahulu.')
if 'v2_audit' not in globals():
    raise RuntimeError('Jalankan audit dataset Step 8 dahulu.')
v2_audit.audit_manifest(REFINED_MANIFEST, ANNOT_JSON, IMAGE_DIR)
step11_record = OUTPUT_A / 'augmentation_training_provenance.json'
v2_audit.audit_augmented_manifest(REFINED_MANIFEST, AUGMENTED_MANIFEST)
if not step11_record.is_file():
    raise FileNotFoundError('Catatan Step 11 augmented belum ada. Jalankan Step 11 baru dahulu.')
step11_protocol = json.loads(step11_record.read_text())
current_hashes = {str(p): hashlib.sha256(p.read_bytes()).hexdigest()
                  for p in (REFINED_MANIFEST, AUGMENTED_MANIFEST, TRAIN_SCRIPT_A, ANNOT_JSON, REFINE_SCRIPT)}
if step11_protocol['sha256'] != current_hashes or step11_protocol['target_ratio'] != 1.0 or step11_protocol['weighted_sampler']:
    raise ValueError('Sumber/protokol berubah dari Step 11. Pulihkan sumber asli atau gunakan eksperimen baru.')
expected_options = {'--image-size': str(IMAGE_SIZE), '--batch-size': str(BATCH_SIZE),
                    '--phase1-epochs': str(PHASE1_EPOCHS), '--phase2-epochs': str(PHASE2_EPOCHS),
                    '--device': DEVICE, '--num-workers': str(min(NUM_WORKERS, 2)),
                    '--class-weighting': 'balanced', '--seed': '42',
                    '--variant': 'proposed', '--label-distribution': 'malds'}
step11_cmd = step11_protocol['command']
if '--weighted-sampler' in step11_cmd or '--pretrained' not in step11_cmd:
    raise ValueError('Konfigurasi sampler/pretrained Step 11 tidak sesuai.')
for flag, value in expected_options.items():
    if flag not in step11_cmd or step11_cmd[step11_cmd.index(flag) + 1] != value:
        raise ValueError(f'Konfigurasi {flag} berbeda dari Step 11.')
ABLATION_DIR = TRAIN_DIR / f'{OUTPUT_A.name}_ablation_aug100_v1'
ablation_df = pd.read_csv(AUGMENTED_MANIFEST, dtype=str, keep_default_na=False)
ablation_df = ablation_df[ablation_df['split'].isin(['train', 'val'])].copy()
ABLATION_MANIFEST = ABLATION_DIR / 'train_val_manifest.csv'
manifest_text = ablation_df.to_csv(index=False)
ablation_protocol = {'step11': step11_protocol, 'variants': ABLATION_VARIANTS,
                     'label_distribution': 'auto', 'skip_test_eval': True,
                     'manifest_sha256': hashlib.sha256(manifest_text.encode()).hexdigest()}
protocol_path = ABLATION_DIR / 'ablation_provenance.json'
if protocol_path.exists():
    if json.loads(protocol_path.read_text()) != ablation_protocol:
        raise ValueError('Protokol ablation berubah; gunakan folder baru, jangan resume run berbeda.')
elif ABLATION_DIR.exists() and any(ABLATION_DIR.iterdir()):
    raise FileExistsError('Folder ablation berisi file tanpa provenance; gunakan folder baru.')
if ABLATION_MANIFEST.exists() and ABLATION_MANIFEST.read_text() != manifest_text:
    raise ValueError('Manifest ablation tersimpan berbeda. Jangan menimpa manifest run lama.')
print('Manifest ablation (tanpa test):', ABLATION_MANIFEST)
print('Variasi:', ABLATION_VARIANTS)
if RUN_ABLATIONS:
    import torch
    ABLATION_DIR.mkdir(parents=True, exist_ok=True)
    if not protocol_path.exists():
        protocol_path.write_text(json.dumps(ablation_protocol, indent=2), encoding='utf-8')
    if not ABLATION_MANIFEST.exists():
        ABLATION_MANIFEST.write_text(manifest_text, encoding='utf-8')
    import re
    import subprocess
    import sys
    import time
    from datetime import timedelta

    ablation_rows = []
    total_epochs = PHASE1_EPOCHS + PHASE2_EPOCHS
    overall_started = time.monotonic()

    def record_ablation_result(variant, run_dir):
        summary = json.loads((run_dir / 'metrics.json').read_text())
        best = min(summary['history'], key=lambda row: row['val_loss'])
        ablation_rows.append({'variant': variant, 'best_epoch': best['epoch'],
                             'val_loss': summary['best_val_loss'], 'train_accuracy': best['train_acc'],
                             'val_accuracy': best['val_acc'],
                             'val_kappa': best['val_kappa'], 'val_dice': best['val_dice'],
                             'val_iou': best['val_iou']})
        pd.DataFrame(ablation_rows).to_csv(ABLATION_DIR / 'ablation_results.csv', index=False)

    for variant_index, variant in enumerate(ABLATION_VARIANTS, start=1):
        run_dir = ABLATION_DIR / variant
        metrics_path = run_dir / 'metrics.json'
        checkpoint_path = run_dir / 'last_checkpoint.pt'
        if metrics_path.exists():
            print(f'✅ Varian {variant_index}/{len(ABLATION_VARIANTS)} ({variant}) sudah selesai; memakai metrics.json yang tersimpan.', flush=True)
            record_ablation_result(variant, run_dir)
            continue
        # Jika runtime berhenti tepat setelah checkpoint epoch terakhir tetapi sebelum metrics.json ditulis,
        # pulihkan ringkasan dari checkpoint agar varian tidak dicoba ulang dari epoch 1.
        if checkpoint_path.exists():
            checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
            if int(checkpoint.get('epoch', 0)) >= total_epochs:
                summary = {'history': checkpoint['history'], 'best_val_loss': checkpoint['best_val_loss'],
                           'test_metrics': None}
                metrics_path.write_text(json.dumps(summary, indent=2))
                print(f'♻️ Checkpoint {variant} sudah mencapai epoch {total_epochs}; metrics.json dipulihkan.', flush=True)
                record_ablation_result(variant, run_dir)
                continue

        cmd = [sys.executable, '-u', str(TRAIN_SCRIPT_A), '--manifest', str(ABLATION_MANIFEST),
               '--output-dir', str(run_dir), '--variant', variant,
               '--class-weighting', 'balanced', '--label-distribution', 'auto',
               '--image-size', str(IMAGE_SIZE), '--batch-size', str(BATCH_SIZE),
               '--phase1-epochs', str(PHASE1_EPOCHS), '--phase2-epochs', str(PHASE2_EPOCHS),
               '--device', DEVICE, '--pretrained', '--num-workers', str(min(NUM_WORKERS, 2)),
               '--seed', '42', '--skip-test-eval', '--resume']
        print(f'\n===== Ablation aug100 {variant_index}/{len(ABLATION_VARIANTS)}: {variant} =====', flush=True)
        print(f'Checkpoint resume aktif jika tersedia: {run_dir / "last_checkpoint.pt"}', flush=True)
        variant_started = time.monotonic()
        epoch_started = None
        epoch_durations = []
        last_epoch = 0
        with subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                              text=True, bufsize=1) as process:
            for line in process.stdout:
                print(line, end='', flush=True)
                if re.match(r'^📘 Epoch \d+/', line):
                    epoch_started = time.monotonic()
                match = re.match(r'^Epoch (\d+) \| train_loss=', line)
                if match:
                    last_epoch = int(match.group(1))
                    if epoch_started is not None:
                        epoch_durations.append(time.monotonic() - epoch_started)
                    if epoch_durations:
                        avg_epoch = sum(epoch_durations) / len(epoch_durations)
                        epochs_left_this_variant = max(0, total_epochs - last_epoch)
                        variants_after = len(ABLATION_VARIANTS) - variant_index
                        eta_seconds = (epochs_left_this_variant + variants_after * total_epochs) * avg_epoch
                        print(f'⏱️ Progress ablation: varian {variant_index}/{len(ABLATION_VARIANTS)} ({variant}), '
                              f'epoch {last_epoch}/{total_epochs}; rata-rata {avg_epoch / 60:.1f} menit/epoch; '
                              f'estimasi sisa sekitar {timedelta(seconds=int(eta_seconds))}.', flush=True)
            return_code = process.wait()
        if return_code != 0:
            raise subprocess.CalledProcessError(return_code, cmd)

        record_ablation_result(variant, run_dir)
        print(f'✅ Varian {variant} selesai dalam {timedelta(seconds=int(time.monotonic() - variant_started))}. '
              f'Ringkasan tersimpan ke {ABLATION_DIR / "ablation_results.csv"}.', flush=True)

    ablation_results = pd.DataFrame(ablation_rows)
    ablation_results.to_csv(ABLATION_DIR / 'ablation_results.csv', index=False)
    print(f'🎉 Semua ablation selesai dalam {timedelta(seconds=int(time.monotonic() - overall_started))}.', flush=True)
    display(ablation_results.sort_values('val_accuracy', ascending=False))
else:
    print('Training OFF. Ubah RUN_ABLATIONS=True untuk tujuh run aug100, termasuk proposed. Hasil lama tidak digunakan.')


---
## 21. Final Training dan Test Set

Jalankan setelah audit filter, cross-validation, dan pemilihan varian selesai. Cell ini melatih model proposed pada split lengkap hasil filter dan mengevaluasi test set satu kali. Folder output terpisah; training lama tidak dihapus. `RUN_FINAL_FLOWCHART` sengaja default `False` agar cell tidak memulai training panjang tanpa disadari.

In [ ]:
FINAL_FLOWCHART_RUN = TRAIN_DIR / '2026-10-24_flowchart_proposed_filtered'
RUN_FINAL_FLOWCHART = False  # Ubah ke True untuk menjalankan/melanjutkan training final dan evaluasi test.
if RUN_FINAL_FLOWCHART:
    import ast
    import re
    import subprocess
    import sys
    import time
    from datetime import timedelta

    # Patch older Colab copies in place: map_location=device moves CUDA RNG state tensors to GPU,
    # but torch.cuda.set_rng_state_all requires CPU ByteTensor values.
    trainer_source = TRAIN_SCRIPT_A.read_text(encoding='utf-8')
    if 'cuda_rng_state_cpu_fix' not in trainer_source:
        old_rng_restore = '            torch.cuda.set_rng_state_all(checkpoint["cuda_rng_state"])'
        new_rng_restore = '''            # cuda_rng_state_cpu_fix: RNG restore requires CPU ByteTensor values.
            cuda_rng_state = checkpoint["cuda_rng_state"]
            if isinstance(cuda_rng_state, torch.Tensor):
                cuda_rng_state = [cuda_rng_state]
            cuda_rng_state = [torch.as_tensor(state, dtype=torch.uint8).cpu().contiguous()
                              for state in cuda_rng_state]
            torch.cuda.set_rng_state_all(cuda_rng_state)'''
        if old_rng_restore not in trainer_source:
            raise RuntimeError(f'Format pemulihan RNG belum dikenali di {TRAIN_SCRIPT_A}; perbarui script training secara manual.')
        trainer_source = trainer_source.replace(old_rng_restore, new_rng_restore, 1)
        ast.parse(trainer_source)
        TRAIN_SCRIPT_A.write_text(trainer_source, encoding='utf-8')
        print(f'🔧 Perbaikan pemulihan RNG diterapkan ke {TRAIN_SCRIPT_A}.', flush=True)

    metrics_path = FINAL_FLOWCHART_RUN / 'metrics.json'
    if metrics_path.exists():
        print(f'✅ Final run sudah selesai; hasil tersimpan di {metrics_path}', flush=True)
    else:
        total_epochs = PHASE1_EPOCHS + PHASE2_EPOCHS
        cmd = [sys.executable, '-u', str(TRAIN_SCRIPT_A), '--manifest', str(FILTERED_MANIFEST),
               '--output-dir', str(FINAL_FLOWCHART_RUN), '--variant', 'proposed',
               '--class-weighting', 'balanced', '--weighted-sampler',
               '--image-size', str(IMAGE_SIZE), '--batch-size', str(BATCH_SIZE),
               '--phase1-epochs', str(PHASE1_EPOCHS), '--phase2-epochs', str(PHASE2_EPOCHS),
               '--device', DEVICE, '--pretrained', '--num-workers', str(min(NUM_WORKERS, 2)),
               '--seed', '42', '--resume']
        checkpoint_path = FINAL_FLOWCHART_RUN / 'last_checkpoint.pt'
        if checkpoint_path.exists():
            print(f'♻️ Checkpoint ditemukan; training akan dilanjutkan: {checkpoint_path}', flush=True)
        else:
            print('🆕 Memulai final training dari epoch pertama.', flush=True)
        print(f'===== Final training: {total_epochs} epoch, lalu evaluasi test =====', flush=True)
        training_started = time.monotonic()
        epoch_started = None
        epoch_durations = []
        last_epoch = 0
        with subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                              text=True, bufsize=1) as process:
            for line in process.stdout:
                print(line, end='', flush=True)
                if re.match(r'^📘 Epoch \d+/', line):
                    epoch_started = time.monotonic()
                match = re.match(r'^Epoch (\d+) \| train_loss=', line)
                if match:
                    last_epoch = int(match.group(1))
                    if epoch_started is not None:
                        epoch_durations.append(time.monotonic() - epoch_started)
                    if epoch_durations:
                        avg_epoch = sum(epoch_durations) / len(epoch_durations)
                        eta_seconds = max(0, total_epochs - last_epoch) * avg_epoch
                        print(f'⏱️ Progress final: epoch {last_epoch}/{total_epochs}; '
                              f'rata-rata {avg_epoch / 60:.1f} menit/epoch; '
                              f'estimasi sisa training sekitar {timedelta(seconds=int(eta_seconds))}.', flush=True)
        return_code = process.wait()
        if return_code != 0:
            raise subprocess.CalledProcessError(return_code, cmd)
        print(f'✅ Final training dan evaluasi selesai dalam '
              f'{timedelta(seconds=int(time.monotonic() - training_started))}.', flush=True)
        print('Final run:', FINAL_FLOWCHART_RUN)
else:
    print('Final training belum dijalankan. Pastikan CV dan ablation selesai, lalu ubah RUN_FINAL_FLOWCHART=True.')


---
## 22. Visualisasi Training dan Confusion Matrix

Setelah final training selesai, jalankan cell ini untuk melihat kurva loss, akurasi, kappa, dan Dice train/validation, ringkasan metrik per fold, serta confusion matrix dan metrik tiap kelas pada test set. Hasil grafik dan prediksi disimpan di folder output final. Cell ini tidak melatih ulang model; confusion matrix dihitung dari checkpoint terbaik final pada test set.


In [ ]:
from pathlib import Path
import importlib.util
from types import SimpleNamespace
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

FINAL_METRICS = FINAL_FLOWCHART_RUN / 'metrics.json'
if not FINAL_METRICS.exists():
    print(f'Final metrics belum ada. Selesaikan final training dahulu: {FINAL_METRICS}')
else:
    final_summary = json.loads(FINAL_METRICS.read_text())
    history = final_summary.get('history', [])
    if not history:
        raise ValueError('metrics.json tidak memiliki riwayat epoch untuk digambar.')

    epochs = [row['epoch'] for row in history]
    best_epoch = min(history, key=lambda row: row['val_loss'])['epoch']
    fig, axes = plt.subplots(2, 2, figsize=(14, 9))
    curves = [
        ('loss', 'Loss'),
        ('acc', 'Accuracy'),
        ('kappa', "Cohen's kappa"),
        ('dice', 'Segmentation Dice'),
    ]
    for ax, (metric, title) in zip(axes.flat, curves):
        for split, label in [('train', 'Train'), ('val', 'Validation')]:
            key = f'{split}_{metric}'
            if key in history[0]:
                ax.plot(epochs, [row.get(key, np.nan) for row in history], label=label)
        ax.axvline(best_epoch, color='gray', linestyle='--', alpha=0.7, label=f'Best val-loss epoch {best_epoch}')
        ax.set_title(title)
        ax.set_xlabel('Epoch')
        ax.grid(alpha=0.25)
        ax.legend()
    fig.suptitle('Final Training Curves', fontsize=15)
    fig.tight_layout()
    curve_path = FINAL_FLOWCHART_RUN / 'training_curves_final.png'
    fig.savefig(curve_path, dpi=140, bbox_inches='tight')
    plt.show()
    print(f'Kurva training tersimpan: {curve_path}')

    # Ringkas variasi hasil CV untuk melihat kestabilan antar-fold.
    cv_csv = FLOWCHART_DIR / 'cross_validation' / 'cross_validation_results.csv'
    if cv_csv.exists():
        cv_results_report = pd.read_csv(cv_csv)
        metric_cols = [col for col in ['val_accuracy', 'val_kappa', 'val_dice', 'val_iou'] if col in cv_results_report]
        display(cv_results_report)
        display(cv_results_report[metric_cols].agg(['mean', 'std']).T.rename(columns={'mean': 'mean', 'std': 'std'}))
        ax = cv_results_report.set_index('fold')[metric_cols].plot(kind='bar', figsize=(10, 5))
        ax.set_title('Cross-validation metrics per fold')
        ax.set_ylabel('Score')
        ax.set_ylim(0, 1)
        ax.grid(axis='y', alpha=0.25)
        plt.xticks(rotation=0)
        plt.tight_layout()
        cv_plot_path = FLOWCHART_DIR / 'cross_validation' / 'cross_validation_metrics.png'
        plt.savefig(cv_plot_path, dpi=140, bbox_inches='tight')
        plt.show()
        print(f'Grafik CV tersimpan: {cv_plot_path}')

    # Muat arsitektur yang sama dan evaluasi prediksi kelas pada test split.
    trainer_spec = importlib.util.spec_from_file_location('acne_hybrid_trainer_report', str(TRAIN_SCRIPT_A))
    trainer = importlib.util.module_from_spec(trainer_spec)
    trainer_spec.loader.exec_module(trainer)
    final_config = final_summary['config']
    final_rows = trainer.load_rows(FILTERED_MANIFEST)
    test_rows = [row for row in final_rows if row['split'] == 'test']
    if not test_rows:
        raise ValueError('Manifest final tidak memiliki split test.')
    dataset_args = SimpleNamespace(resolved_input_normalization=final_config.get('resolved_input_normalization', 'imagenet'))
    test_dataset = trainer.PreprocessedAcneDataset(
        test_rows, image_size=int(final_config['image_size']), augment=False, args=dataset_args
    )
    test_loader = torch.utils.data.DataLoader(
        test_dataset, batch_size=int(final_config['batch_size']), shuffle=False,
        num_workers=min(int(final_config.get('num_workers', 2)), 2)
    )
    device = trainer.resolve_device(DEVICE)
    model = trainer.ProposalHybridResNet50(num_classes=4, pretrained=False, variant='proposed').to(device)
    best_weights = torch.load(FINAL_FLOWCHART_RUN / 'best_model.pt', map_location=device, weights_only=True)
    model.load_state_dict(best_weights)
    model.eval()
    y_true, y_pred = [], []
    with torch.inference_mode():
        for batch in test_loader:
            outputs = model(batch['image'].to(device))
            y_true.extend(batch['label'].tolist())
            y_pred.extend(outputs['class_logits'].argmax(dim=1).cpu().tolist())

    class_names = trainer.LABEL_NAMES
    cm = confusion_matrix(y_true, y_pred, labels=list(range(len(class_names))))
    cm_norm = confusion_matrix(y_true, y_pred, labels=list(range(len(class_names))), normalize='true')
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    ConfusionMatrixDisplay(cm, display_labels=class_names).plot(ax=axes[0], colorbar=False, cmap='Blues', values_format='d')
    axes[0].set_title('Test confusion matrix (count)')
    ConfusionMatrixDisplay(cm_norm, display_labels=class_names).plot(ax=axes[1], colorbar=False, cmap='Blues', values_format='.2f')
    axes[1].set_title('Test confusion matrix (row normalized)')
    fig.tight_layout()
    cm_path = FINAL_FLOWCHART_RUN / 'confusion_matrix_test.png'
    fig.savefig(cm_path, dpi=140, bbox_inches='tight')
    plt.show()
    print(f'Confusion matrix tersimpan: {cm_path}')

    report = classification_report(y_true, y_pred, labels=list(range(len(class_names))),
                                   target_names=class_names, output_dict=True, zero_division=0)
    display(pd.DataFrame(report).T)
    predictions_path = FINAL_FLOWCHART_RUN / 'test_class_predictions.csv'
    pd.DataFrame({
        'filename': [row['filename'] for row in test_rows],
        'true_label': [class_names[label] for label in y_true],
        'predicted_label': [class_names[label] for label in y_pred],
    }).to_csv(predictions_path, index=False)
    print(f'Prediksi per foto tersimpan: {predictions_path}')


---
## Troubleshooting

| Masalah | Solusi |
|---|---|
| `CUDA not available` | Runtime → Change runtime type → GPU |
| `CUDA out of memory` | Kurangi `BATCH_SIZE` ke `8` atau `IMAGE_SIZE` ke `224` |
| `NameError: Path` | Jalankan ulang cell **Import Python Libraries** (cell 4) |
| `ModuleNotFoundError` | Jalankan ulang cell Install Dependencies + Import |
| Dataset tidak terdownload | Pastikan link folder Drive bisa diakses publik, lalu jalankan ulang cell 5-6 |
| Scripts zip tidak ditemukan | Upload `acne_scripts.zip` ke `MyDrive/acne_project/` lalu jalankan ulang cell 7 |
| Sesi putus saat training | Jalankan ulang cell 2-8, lalu training dengan flag `--resume` |
| Training sangat lambat | Naikkan `NUM_WORKERS` ke `4`, atau upgrade ke Colab Pro |
| Dice masih 0 setelah training | Naikkan `focal_alpha` ke `0.85`, gunakan `--weighted-sampler` |

### Target metrik realistis (1204 gambar + T4 GPU)
| Metrik | Target |
|---|---|
| Dice | > 0.05 |
| IoU | > 0.03 |
| Accuracy | > 0.40 |
| Kappa | > 0.10 |


---
## Penanda Update - 24 Oktober 2026

**Versi: `2026-10-24-training-pipeline-v2`**

Catatan ini menandai pembaruan pipeline preprocessing, training, dan penambahan jalur eksperimen flowchart opsional. Jika terlihat di bagian paling bawah Colab, notebook yang dibuka sudah memuat penanda ini.

- Preprocessing default sekarang tidak melakukan equalize warna atau sharpen; resize default memakai letterbox untuk menjaga rasio aspek. Mode lama tetap tersedia untuk eksperimen pembanding.
- Optimizer dipertahankan per fase, dan checkpoint menyimpan state optimizer serta random state untuk resume.
- Saat memakai `--pretrained`, normalisasi ImageNet digunakan otomatis; inferensi membaca konfigurasi yang tersimpan di `metrics.json`.
- Notebook memakai folder output baru `refined_masks_output_letterbox_v2` dan `2026-10-24_proposal_hybrid_letterbox_v2_{CLASS_WEIGHTING}`. Training ulang belum dijalankan.
- Steps 18–21 menambahkan audit foto/mask manual, 5-fold CV pada train split, ablation (baseline, no-attention, no-dilation, no-AMFM, LDS, no-fusion), dan final run flowchart. Jalur ini opsional; flag eksekusi training default `False` dan tiap eksperimen menulis ke folder baru. Training belum dijalankan.
- Ablation baseline mempertahankan decoder segmentasi bersama agar metrik Dice/IoU tetap tersedia; jadi baseline ini bukan ResNet klasifikasi murni.

### Update lanjutan — 24 Oktober 2026
- Rencana opsi eksperimen baseline: sediakan pilihan `hybrid_baseline` (baseline saat ini, tetap menghitung segmentasi) dan `resnet50_classifier` (ResNet-50 klasifikasi murni), agar keduanya bisa dibandingkan. Opsi klasifikasi murni belum diimplementasikan atau dilatih; jangan pilih varian tersebut sebelum script training diperbarui.
- Perubahan notebook ini tidak otomatis memperbarui `acne_scripts.zip` di Google Drive. Cell Step 18 mencoba mengambil trainer terbaru dari GitHub bila script Drive belum mendukung opsi baru. Commit/push script terbaru sebelum menjalankan. Jangan resume checkpoint lama untuk run baru.

Perubahan lokal harus di-commit dan push sebelum penanda ini muncul saat notebook dibuka ulang dari GitHub.


---
## 23. Eksperimen SMOTE untuk Klasifikasi (opsional)
Jalankan setelah training biasa selesai. Checkpoint hibrid dibekukan sebagai ekstraktor fitur; SMOTE moderat hanya diterapkan pada fitur klasifikasi train, lalu classification head dilatih ulang. Foto-mask asli tetap melatih/evaluasi segmentasi. Validation dan test tidak di-resample. Baca `docs/SMOTE_EXPERIMENT.md` untuk metode dan batas interpretasi.


In [ ]:
SMOTE_RUN = TRAIN_DIR / 'smote_feature_experiment_120epochs'
SMOTE_MANIFEST = REFINED_MANIFEST  # Untuk jalur flowchart, ganti ke FILTERED_MANIFEST.
SMOTE_BASELINE = OUTPUT_A  # Harus berisi best_model.pt dan metrics.json dari training biasa.
SMOTE_SCRIPT = SCRIPTS_DIR / 'train_smote_classifier_head.py'

if not (SMOTE_BASELINE / 'best_model.pt').exists():
    raise FileNotFoundError(f'Checkpoint baseline tidak ditemukan: {SMOTE_BASELINE}')
if not (SMOTE_BASELINE / 'metrics.json').exists():
    raise FileNotFoundError('metrics.json baseline tidak ditemukan.')
from urllib.request import urlopen
SMOTE_SCRIPT_URL = 'https://raw.githubusercontent.com/Pandeigede2803/acne_segmentation_github/main/scripts/train_smote_classifier_head.py'
TRAINER_URL = 'https://raw.githubusercontent.com/Pandeigede2803/acne_segmentation_github/main/scripts/train_proposal_hybrid_resnet50_malds.py'
with urlopen(SMOTE_SCRIPT_URL, timeout=60) as response:
    SMOTE_SCRIPT.write_bytes(response.read())
with urlopen(TRAINER_URL, timeout=60) as response:
    TRAIN_SCRIPT_A.write_bytes(response.read())
print('Script SMOTE dan trainer terbaru diunduh dari GitHub.')

!python "{SMOTE_SCRIPT}" \
  --trainer "{TRAIN_SCRIPT_A}" \
  --manifest "{SMOTE_MANIFEST}" \
  --checkpoint "{SMOTE_BASELINE / 'best_model.pt'}" \
  --metrics "{SMOTE_BASELINE / 'metrics.json'}" \
  --output-dir "{SMOTE_RUN}" \
  --epochs 120 \
  --device {DEVICE}


## 24. Visualisasi Hasil Training SMOTE
Jalankan setelah Step 23. Cell ini hanya membaca hasil tersimpan, tidak melatih ulang. Kurva akurasi dan loss train memakai fitur train asli (bukan sintetis). Chart meliputi akurasi/loss train–validasi, distribusi kelas, confusion matrix test (jumlah dan normalisasi per kelas sebenarnya), serta precision/recall/F1 per kelas. PNG dan tabel CSV disimpan di `SMOTE_RUN / 'charts'`. Confusion matrix memerlukan run selesai dengan script terbaru; run lama atau parsial hanya menampilkan data yang tersedia. Pemilihan epoch terbaik tetap berdasarkan validation loss, bukan test.


In [ ]:
from urllib.request import urlopen
from IPython.display import Image, display
import subprocess
import sys

# Jika runtime baru, definisikan ulang SMOTE_RUN ke folder hasil Step 23 di Drive.
if 'SMOTE_RUN' not in globals():
    raise RuntimeError('Definisikan SMOTE_RUN ke folder hasil training terlebih dahulu.')
PLOT_SCRIPT = SCRIPTS_DIR / 'plot_smote_results.py'
PLOT_URL = 'https://raw.githubusercontent.com/Pandeigede2803/acne_segmentation_github/main/scripts/plot_smote_results.py'
with urlopen(PLOT_URL, timeout=60) as response:
    PLOT_SCRIPT.write_bytes(response.read())
subprocess.run([sys.executable, str(PLOT_SCRIPT), '--run-dir', str(SMOTE_RUN)], check=True)
CHART_DIR = SMOTE_RUN / 'charts'
for chart in sorted(CHART_DIR.glob('*.png')):
    print(chart.name)
    display(Image(filename=str(chart)))
print('PNG dan CSV tersimpan di:', CHART_DIR)


## 25. Audit Kesalahan Severity pada Validation (tanpa training)
Gunakan backbone, manifest, dan head terbaik yang berpasangan dari Step 23. Tidak ada training dan tidak ada evaluasi test. Audit menampilkan confusion matrix jumlah/normalisasi, support dan error rate per kelas, pasangan severity yang sering tertukar, serta contoh foto salah prediksi. Contoh dipilih per pasangan berdasarkan confidence tertinggi, jadi bukan sampel representatif. Lihat jumlah **dan** proporsi kesalahan untuk membedakan masalah umum dari masalah kelas tertentu; confidence bukan ukuran kepastian klinis. CSV menyimpan semua kesalahan dan jalur foto. Output lokal/Drive dapat berisi foto sensitif: jangan commit/publikasikan tanpa izin.


In [ ]:
import json
import subprocess
import sys
from urllib.request import urlopen
from IPython.display import Image, display

AUDIT_HEAD = SMOTE_RUN / 'smote_classifier_head.pt'
AUDIT_DIR = SMOTE_RUN / 'validation_audit'
if not AUDIT_HEAD.exists():
    raise FileNotFoundError(f'Head terbaik belum tersedia: {AUDIT_HEAD}')
for name in ['audit_smote_validation.py', 'train_proposal_hybrid_resnet50_malds.py']:
    url = f'https://raw.githubusercontent.com/Pandeigede2803/acne_segmentation_github/main/scripts/{name}'
    with urlopen(url, timeout=60) as response:
        (SCRIPTS_DIR / name).write_bytes(response.read())
subprocess.run([sys.executable, str(SCRIPTS_DIR / 'audit_smote_validation.py'),
    '--trainer', str(TRAIN_SCRIPT_A), '--manifest', str(SMOTE_MANIFEST),
    '--checkpoint', str(SMOTE_BASELINE / 'best_model.pt'),
    '--metrics', str(SMOTE_BASELINE / 'metrics.json'), '--head', str(AUDIT_HEAD),
    '--output-dir', str(AUDIT_DIR), '--device', str(DEVICE),
    '--examples-per-pair', '3', '--max-examples', '24'], check=True)
audit = json.loads((AUDIT_DIR / 'validation_summary.json').read_text())
print('Validation accuracy:', audit['accuracy'])
display(audit['per_class'])
print('Pasangan severity yang paling sering tertukar:')
display(audit['confused_pairs'])
for name in ['validation_confusion.png', 'validation_confusion_normalized.png', 'validation_error_examples.png']:
    if (AUDIT_DIR / name).exists():
        display(Image(filename=str(AUDIT_DIR / name)))
print('Audit dan CSV:', AUDIT_DIR)


## 26. Percobaan Dropout 0,3 → 0,5 (hidden tetap 64)
Default tidak melatih. Setelah audit, aktifkan flag yang diperlukan. Run kontrol (64, dropout 0,3) dan run dropout (64, dropout 0,5) memakai backbone/manifest/SMOTE/seed/batch size/120 epoch yang sama, head dilatih dari awal, bukan resume. Run lama belum tentu memiliki inisialisasi terkontrol, jadi sediakan kontrol baru untuk perbandingan. Hanya dropout berubah terhadap kontrol; perbedaan operasi dropout tetap dapat mengubah lintasan RNG. Seed membantu keterulangan tetapi tidak menjamin hasil bit-identik di semua perangkat. Test **tidak diakses** (`--skip-test`). Tiap run memakai folder baru; folder berisi hasil tidak ditimpa. Ini percobaan, bukan jaminan peningkatan.


In [ ]:
import json
import subprocess
import sys
from urllib.request import urlopen

RUN_CONTROL = False
RUN_DROPOUT_05 = False
REGULARIZATION_ROOT = TRAIN_DIR / 'smote_regularization_v1'  # Ganti suffix untuk run ulang.
REG_EPOCHS, REG_SEED, REG_BATCH_SIZE = 120, 42, 64
REG_RUNS = {
    'control': (64, 0.3),
    'dropout_05': (64, 0.5),
    'hidden_32': (32, 0.3),
}

def run_severity_experiment(name):
    for script_name in ['train_smote_classifier_head.py', 'train_proposal_hybrid_resnet50_malds.py', 'plot_smote_results.py']:
        url = f'https://raw.githubusercontent.com/Pandeigede2803/acne_segmentation_github/main/scripts/{script_name}'
        with urlopen(url, timeout=60) as response:
            (SCRIPTS_DIR / script_name).write_bytes(response.read())
    hidden, dropout = REG_RUNS[name]
    run_dir = REGULARIZATION_ROOT / name
    command = [sys.executable, str(SCRIPTS_DIR / 'train_smote_classifier_head.py'),
        '--trainer', str(TRAIN_SCRIPT_A), '--manifest', str(SMOTE_MANIFEST),
        '--checkpoint', str(SMOTE_BASELINE / 'best_model.pt'),
        '--metrics', str(SMOTE_BASELINE / 'metrics.json'), '--output-dir', str(run_dir),
        '--epochs', str(REG_EPOCHS), '--seed', str(REG_SEED), '--batch-size', str(REG_BATCH_SIZE),
        '--hidden-dim', str(hidden), '--dropout', str(dropout), '--device', str(DEVICE), '--skip-test']
    subprocess.run(command, check=True)
    subprocess.run([sys.executable, str(SCRIPTS_DIR / 'plot_smote_results.py'),
                    '--run-dir', str(run_dir)], check=True)

def show_severity_comparison():
    rows = []
    for name in REG_RUNS:
        path = REGULARIZATION_ROOT / name / 'metrics.json'
        if not path.exists():
            print(name, ': belum dijalankan')
            continue
        result = json.loads(path.read_text())
        best = min(result['history'], key=lambda row: row['val_loss'])
        rows.append({'run': name, 'hidden_dim': result['config']['hidden_dim'],
                     'dropout': result['config']['dropout'], 'best_epoch': best['epoch'],
                     'val_loss': best['val_loss'], 'val_acc': best['val_acc'],
                     'train_acc': best['train_acc'], 'acc_gap': best['train_acc'] - best['val_acc']})
    display(rows)
    print('Bandingkan validation loss/akurasi dan audit per kelas; gap lebih kecil saja belum berarti lebih baik.')

if RUN_CONTROL:
    run_severity_experiment('control')
if RUN_DROPOUT_05:
    run_severity_experiment('dropout_05')
show_severity_comparison()


## 27. Percobaan Hidden Layer 64 → 32 (dropout tetap 0,3)
Jalankan cell konfigurasi Step 26 terlebih dahulu; flag training-nya boleh tetap False. Ini percobaan terpisah, bukan gabungan dengan dropout 0,5. Backbone dan pengaturan lain tetap sama dengan kontrol, head baru lebih kecil. Default tidak melatih dan test tidak diakses. Untuk audit head eksperimen tanpa training, di Step 25 ganti `AUDIT_HEAD` dan `AUDIT_DIR` ke folder run tersebut (manifest/backbone tetap sama). Pilih konfigurasi dari validasi; simpan test untuk evaluasi final setelah pilihan dikunci. Hasil validasi dapat optimistis jika dipakai berulang kali untuk tuning; konfirmasi temuan dengan beberapa seed atau CV bila memungkinkan.


In [ ]:
RUN_HIDDEN_32 = False
if 'run_severity_experiment' not in globals():
    raise RuntimeError('Jalankan cell konfigurasi Step 26 terlebih dahulu.')
if RUN_HIDDEN_32:
    run_severity_experiment('hidden_32')
show_severity_comparison()


## 28. Variasi tambahan fine-tuning ResNet50 layer4 + chart (validation only)
**Posisi penelitian:** eksperimen tambahan, bukan pengganti metode utama proposal. Arsitektur hibrid tetap, tetapi protokol variasi memakai AdamW dan CE tanpa MA-LDS/morphology consistency loss/weighted sampler. Laporkan hasil terpisah dari metode utama; tiga seed tidak menggantikan 5-fold CV. Resume/log hanya fasilitas teknis.
**Satu step, dua cell:** training + ringkasan (ringkasan bisa diulang tanpa training). Step 1–27 tidak diubah. Default training OFF. Periksa FT_MANIFEST/FT_BASELINE: sumber SMOTE jika tersedia, jika tidak REFINED_MANIFEST/OUTPUT_A. Tidak perlu menjalankan training Step 26/27.

**Perbandingan:** `frozen` (classifier saja) vs `layer4` (classifier + layer4), mulai dari checkpoint backbone/classifier terlatih yang sama. Dropout 0,3; tanpa SMOTE/augmentasi online/class weighting; BatchNorm tetap eval. Modul lain dibekukan. Bandingkan terhadap kontrol baru, bukan langsung hasil 71,30% SMOTE karena protokol berbeda.

**Training:** LR layer4 1e-5 / classifier 1e-4; maksimal 30 epoch, patience 8. Loss = classification CE + segmentation BCE + soft Dice loss. Checkpoint dipilih dari validation classification CE (epoch 0 dipertahankan jika tidak membaik). Dice/IoU terhadap mask manifest dipantau; auxiliary loss tidak menjamin segmentasi bebas regresi.

**Durasi & log:** 2 konfigurasi × 3 seed (42,43,44) = **6 run berurutan**, masing-masing maksimal 30 epoch. Log batch/epoch tampil langsung dan tersimpan di `FT_ROOT/<mode>_seed<seed>.log` (termasuk error training). Untuk smoke test, gunakan FT_SEEDS=[42] dan FT_EPOCHS=1 di folder FT_ROOT berbeda; bukan hasil final.

**Resume otomatis:** FT_RESUME=True. Simpan **FT_ROOT di Google Drive**, bukan disk sementara `/content`, agar checkpoint bertahan saat runtime hilang. Setelah reconnect, mount Drive, pulihkan variabel/path/dependensi, lalu jalankan ulang cell dengan flag training aktif. Run yang memiliki `last_checkpoint.pt` dilanjutkan dari akhir epoch terakhir; run completed dilewati dan run yang belum dimulai dijalankan. Jika putus di tengah epoch, hanya epoch itu diulang. Model, optimizer, RNG, shuffle DataLoader, history dan patience disimpan secara atomik tiap akhir epoch (termasuk epoch 0); checkpoint terbaik/prediksi dapat dipulihkan dari checkpoint terakhir. Log resume ditambahkan, tidak ditimpa. Tidak ada jaminan bit-identik jika perangkat/versi library berubah.

**Batas resume:** konfigurasi, path sumber dan SHA-256 manifest/checkpoint/metrics/trainer harus sama; jangan ubah batch/seed/epoch/LR saat resume. File gambar/mask harus tetap sama (kontennya tidak di-hash). Run lama dari versi tanpa `last_checkpoint.pt` tidak bisa di-resume. Untuk konfigurasi baru gunakan FT_ROOT baru; FT_RESUME=False menolak folder/log lama. Jangan jalankan dua proses untuk run yang sama.

**Evaluasi:** mean±sample SD, delta berpasangan, macro-F1/recall serta Dice/IoU; tiga seed bukan bukti signifikansi statistik. Hanya train/val, **tidak test**. Pastikan split subject/source bebas kebocoran dan checkpoint belum dilatih pada val/test. File hasil run yang sama diperbarui saat resume; checkpoint awal training biasa tidak diubah.

**Chart (cell ringkasan):** kurva train total loss dan validation CE pada panel berbeda (bukan loss yang setara), kurva validation accuracy/macro-F1 dan Dice/IoU, confusion matrix val (jumlah & proporsi), recall per kelas; perbandingan antar-seed mean±sample SD dan delta berpasangan hanya untuk run completed. Kurva partial diberi label running. Tidak ada angka rekaan atau akses test. Grafik per-run di `<run>/charts/`, grafik perbandingan dan CSV di `FT_ROOT/charts/`. Grafik dibuat ulang dari metrics.json, tidak melatih ulang; kurva train accuracy belum tersedia.

Script `train_layer4_finetune.py` dan `plot_layer4_results.py` perlu tersedia di GitHub sebelum unduhan Colab; alternatif unggah keduanya ke SCRIPTS_DIR dan set FT_DOWNLOAD=False. Plot memerlukan matplotlib (umumnya tersedia di Colab). Tidak ada jaminan performa meningkat.


In [ ]:
import json
import statistics
import subprocess
import sys
from pathlib import Path
from urllib.request import urlopen

RUN_FT_CONTROL = False
RUN_FT_LAYER4 = False
FT_RESUME = True  # Jalankan ulang untuk melanjutkan run yang terputus.
FT_DOWNLOAD = True
FT_SEEDS = [42, 43, 44]
FT_EPOCHS, FT_PATIENCE, FT_BATCH_SIZE = 30, 8, 8  # Jika OOM, kecilkan batch untuk KEDUA arm.
# Fallback aman: argumen default globals().get dievaluasi lebih dulu oleh Python.
_refined_manifest = globals().get('REFINED_MANIFEST')
_output_a = globals().get('OUTPUT_A')
_train_dir = globals().get('TRAIN_DIR')
if _refined_manifest is None or _output_a is None or _train_dir is None:
    raise RuntimeError('Setup belum dijalankan. Setelah runtime reset, jalankan mount Drive, cell konfigurasi pusat yang mendefinisikan REFINED_MANIFEST/TRAIN_DIR (cell 18), preprocessing sampai manifest tersedia, lalu cell baseline yang mendefinisikan OUTPUT_A (cell 28). Setelah itu jalankan Step 28.')
FT_MANIFEST = Path(globals().get('SMOTE_MANIFEST') or _refined_manifest)
FT_BASELINE = Path(globals().get('SMOTE_BASELINE') or _output_a)
FT_ROOT = Path(_train_dir) / 'layer4_finetune_v1'
FT_SCRIPT = SCRIPTS_DIR / 'train_layer4_finetune.py'

print('Manifest:', FT_MANIFEST, '\nCheckpoint:', FT_BASELINE, '\nOutput:', FT_ROOT, '\nResume:', FT_RESUME)
if str(FT_ROOT).startswith('/content/') and not str(FT_ROOT).startswith('/content/drive/'):
    print('PERINGATAN: FT_ROOT di disk sementara Colab. Gunakan folder Google Drive untuk tahan runtime reset.')

if RUN_FT_CONTROL or RUN_FT_LAYER4:
    for path in [FT_MANIFEST, FT_BASELINE / 'best_model.pt', FT_BASELINE / 'metrics.json']:
        if not path.is_file():
            raise FileNotFoundError(path)
    if FT_DOWNLOAD:
        url = 'https://raw.githubusercontent.com/Pandeigede2803/acne_segmentation_github/main/scripts/train_layer4_finetune.py'
        with urlopen(url, timeout=60) as response:
            FT_SCRIPT.write_bytes(response.read())
        # The zip may contain an old trainer without the `variant` constructor argument.
        trainer_url = 'https://raw.githubusercontent.com/Pandeigede2803/acne_segmentation_github/main/scripts/train_proposal_hybrid_resnet50_malds.py'
        with urlopen(trainer_url, timeout=60) as response:
            TRAIN_SCRIPT_A.write_bytes(response.read())
        print('Trainer dan script variasi terbaru diunduh dari GitHub.', flush=True)
    if not FT_SCRIPT.is_file():
        raise FileNotFoundError(f'Upload script baru ke {FT_SCRIPT}')
    for mode, enabled in [('frozen', RUN_FT_CONTROL), ('layer4', RUN_FT_LAYER4)]:
        if not enabled:
            continue
        for seed in FT_SEEDS:
            run_dir = FT_ROOT / f'{mode}_seed{seed}'
            command = [sys.executable, str(FT_SCRIPT), '--trainer', str(TRAIN_SCRIPT_A),
                       '--manifest', str(FT_MANIFEST), '--checkpoint', str(FT_BASELINE / 'best_model.pt'),
                       '--metrics', str(FT_BASELINE / 'metrics.json'), '--output-dir', str(run_dir),
                       '--mode', mode, '--seed', str(seed), '--epochs', str(FT_EPOCHS),
                       '--patience', str(FT_PATIENCE), '--batch-size', str(FT_BATCH_SIZE),
                       '--backbone-lr', '1e-5', '--head-lr', '1e-4', '--seg-weight', '1.0',
                       '--device', str(DEVICE)]
            status_path = run_dir / 'metrics.json'
            if FT_RESUME and status_path.is_file():
                try:
                    existing_status = json.loads(status_path.read_text()).get('status')
                except (OSError, json.JSONDecodeError):
                    existing_status = None
                if existing_status == 'completed':
                    print(f'SKIP {mode}, seed={seed}: sudah completed.', flush=True)
                    continue
            has_checkpoint = (run_dir / 'last_checkpoint.pt').is_file()
            if FT_RESUME and has_checkpoint:
                command.append('--resume')
            elif run_dir.exists() and any(run_dir.iterdir()):
                raise FileExistsError(f'Folder berisi hasil tanpa resume valid; periksa atau ganti FT_ROOT: {run_dir}')
            FT_ROOT.mkdir(parents=True, exist_ok=True)
            log_path = FT_ROOT / f'{mode}_seed{seed}.log'
            with log_path.open('a' if FT_RESUME else 'x', encoding='utf-8') as log:
                header = f'RUN {mode}, seed={seed} | log={log_path}\nCommand: {command}\n'
                print(header, flush=True)
                log.write(header)
                log.flush()
                with subprocess.Popen([command[0], '-u', *command[1:]], stdout=subprocess.PIPE,
                                      stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
                    for line in process.stdout:
                        print(line, end='', flush=True)
                        log.write(line)
                        log.flush()
                    return_code = process.wait()
                log.write(f'Exit code: {return_code}\n')
            if return_code:
                raise RuntimeError(f'Training gagal (exit {return_code}). Cek {log_path}')
else:
    print('Training OFF. Periksa sumber, lalu aktifkan RUN_FT_CONTROL dan RUN_FT_LAYER4.')


In [ ]:
# Ringkasan checkpoint terpilih; boleh dijalankan ulang tanpa training.
ft_results = {}
ft_rows = []
for mode in ['frozen', 'layer4']:
    for seed in FT_SEEDS:
        path = FT_ROOT / f'{mode}_seed{seed}' / 'metrics.json'
        if not path.exists():
            print(mode, seed, ': belum tersedia')
            continue
        result = json.loads(path.read_text())
        if result.get('status') != 'completed':
            print(mode, seed, ': run belum selesai, tidak masuk ringkasan')
            continue
        best = result['best_validation']
        ft_results[mode, seed] = best
        ft_rows.append({'mode': mode, 'seed': seed, 'best_epoch': result['best_epoch'], **best,
                        'dice_change_from_initial': best['val_dice'] - result['initial_validation']['val_dice']})
display(ft_rows)
for metric in ['val_loss', 'accuracy', 'macro_f1', 'val_dice', 'val_iou']:
    print('\n' + metric + ' (accuracy/F1/Dice/IoU skala 0–1)')
    for mode in ['frozen', 'layer4']:
        values = [ft_results[mode, s][metric] for s in FT_SEEDS if (mode, s) in ft_results]
        if values:
            sd = f'{statistics.stdev(values):.4f}' if len(values) > 1 else 'N/A'
            print(f'{mode}: mean={statistics.mean(values):.4f}, sample SD={sd}, n={len(values)}')
    paired = [ft_results['layer4', s][metric] - ft_results['frozen', s][metric]
              for s in FT_SEEDS if ('layer4', s) in ft_results and ('frozen', s) in ft_results]
    if paired:
        print('Delta layer4 - frozen per seed:', paired, '; mean:', statistics.mean(paired))
print('Recall per kelas dan confusion matrix ada pada tabel/metrics.json; prediksi val ada di val_predictions.csv.')
print('Jangan memilih hanya dari accuracy; cek macro-F1, recall, Dice/IoU dan konsistensi seed. Test tidak digunakan.')

# Chart variasi tambahan dari file hasil; tidak menjalankan training.
FT_PLOTS = True
if FT_PLOTS and any((FT_ROOT / f'{m}_seed{s}' / 'metrics.json').exists()
                    for m in ['frozen', 'layer4'] for s in FT_SEEDS):
    ft_plot_script = SCRIPTS_DIR / 'plot_layer4_results.py'
    if FT_DOWNLOAD:
        url = 'https://raw.githubusercontent.com/Pandeigede2803/acne_segmentation_github/main/scripts/plot_layer4_results.py'
        with urlopen(url, timeout=60) as response:
            ft_plot_script.write_bytes(response.read())
    if not ft_plot_script.is_file():
        raise FileNotFoundError(f'Upload script chart ke {ft_plot_script}')
    subprocess.run([sys.executable, str(ft_plot_script), '--root', str(FT_ROOT),
                    '--seeds', *map(str, FT_SEEDS)], check=True)
    from IPython.display import Image as FTChartImage
    previews = [FT_ROOT / 'charts/seed_comparison.png', FT_ROOT / 'charts/paired_deltas.png']
    previews += [FT_ROOT / f'{m}_seed{FT_SEEDS[0]}' / 'charts/learning_curves.png'
                 for m in ['frozen', 'layer4']]
    for image_path in previews:
        if image_path.is_file():
            display(FTChartImage(filename=str(image_path), width=950))
else:
    print('Chart OFF atau belum ada hasil training; tidak membuat angka contoh.')


## 29. Pembanding Single-Task ResNet-50 — Augmentasi 100%
Jalankan langsung setelah setup Step 10a/11 (atau setelah Step 20); **tidak perlu menjalankan Step 21–28**. Step baru ini ditaruh di akhir agar penomoran lama tidak berubah. Tidak mengubah training proposed yang sedang berjalan. Jangan menjalankan kedua training bersamaan pada GPU yang sama.

**Model:** ResNet-50 ImageNet → GAP → proyeksi 128 → classifier 64/dropout 0,3 → empat kelas. Tidak ada decoder, Spatial Attention, AMFM, mask input, MA-LDS, atau loss segmentasi/morfologi. Dataset image-only, loss CE berbobot per sampel. Ini lebih ringan dari hybrid, bukan jaminan training cepat. Anggaran default tetap 30 freeze + 90 fine-tuning agar sebanding, bukan dipangkas untuk mengejar skor.

**Protokol:** train/validation dari manifest augmented Step 11 yang sama, balanced class weighting, tanpa sampler, seed 42, preprocessing dan augmentasi online sama. Test dibuang sebelum training. Checkpoint single-task dipilih berdasarkan validation CE minimum; proposed memakai total multi-task validation loss. Perbandingan ini menilai sistem keseluruhan, bukan efek segmentasi saja karena objective dan arsitektur berbeda. Gunakan ablation untuk analisis komponen. Satu seed bukan bukti signifikansi statistik.

**Pengaman:** default training OFF. Folder terpisah; checkpoint tiap akhir epoch menyimpan optimizer/RNG/history serta state terbaik secara atomik. Setelah reconnect pulihkan OUTPUT_A dan sumber Step 11, lalu jalankan ulang dengan flag aktif untuk resume. Folder/config/hash harus sama; untuk pengulangan baru ganti suffix ST_BUNDLE. Isi piksel sumber tidak di-hash; jangan ubah foto/mask. Unggah `train_single_task_resnet50.py` ke SCRIPTS_DIR atau publish ke GitHub sebelum unduhan pertama. Tidak mengambil bobot dari proposed yang sudah dilatih.

In [ ]:
import json, hashlib, subprocess, sys
from pathlib import Path
from urllib.request import urlopen
import pandas as pd
RUN_SINGLE_TASK = False
if 'OUTPUT_A' not in globals() or 'TRAIN_DIR' not in globals():
    raise RuntimeError('Pulihkan konfigurasi OUTPUT_A/TRAIN_DIR dari Step 11 augmented dahulu.')
ST_BUNDLE = TRAIN_DIR / f'{OUTPUT_A.name}_single_task_v1'
ST_RUN = ST_BUNDLE / 'run'
ST_SCRIPT = SCRIPTS_DIR / 'train_single_task_resnet50.py'
if RUN_SINGLE_TASK:
    source_record = OUTPUT_A / 'augmentation_training_provenance.json'
    source_protocol = json.loads(source_record.read_text())
    v2_audit.audit_manifest(REFINED_MANIFEST, ANNOT_JSON, IMAGE_DIR)
    v2_audit.audit_augmented_manifest(REFINED_MANIFEST, AUGMENTED_MANIFEST)
    hashes = {str(p): v2_audit.sha256(p) for p in (REFINED_MANIFEST, AUGMENTED_MANIFEST, TRAIN_SCRIPT_A, ANNOT_JSON, REFINE_SCRIPT)}
    if hashes != source_protocol['sha256'] or source_protocol['weighted_sampler'] or source_protocol['target_ratio'] != 1.0:
        raise ValueError('Sumber/protokol berbeda dari Step 11 augmented.')
    if not ST_SCRIPT.is_file():
        with urlopen('https://raw.githubusercontent.com/Pandeigede2803/acne_segmentation_github/main/scripts/train_single_task_resnet50.py', timeout=60) as response:
            ST_SCRIPT.write_bytes(response.read())
    source_cmd = source_protocol['command']
    def source_option(flag, default):
        return source_cmd[source_cmd.index(flag) + 1] if flag in source_cmd else default
    if source_option('--variant', '') != 'proposed' or source_option('--label-distribution', '') != 'malds':
        raise ValueError('Sumber harus proposed lengkap dari Step 11.')
    if source_option('--class-weighting', '') != 'balanced' or '--pretrained' not in source_cmd or '--weighted-sampler' in source_cmd:
        raise ValueError('Kontrol mengharuskan protokol balanced, pretrained, tanpa sampler seperti Step 11.')
    image_rows = pd.read_csv(AUGMENTED_MANIFEST, dtype=str, keep_default_na=False)
    image_rows = image_rows[image_rows['split'].isin(['train', 'val'])]
    image_rows = image_rows[['filename', 'split', 'label_index', 'image_path', 'preprocessed_path']]
    manifest_text = image_rows.to_csv(index=False)
    st_manifest = ST_BUNDLE / 'train_val_images.csv'
    command = [sys.executable, '-u', str(ST_SCRIPT), '--manifest', str(st_manifest), '--output-dir', str(ST_RUN), '--pretrained']
    for flag, default in {'--image-size': '320', '--batch-size': '16', '--phase1-epochs': '30', '--phase2-epochs': '90',
                          '--phase1-lr': '0.001', '--phase2-lr': '0.0001', '--rotation-deg': '15', '--hflip-prob': '0.5',
                          '--brightness': '0.15', '--contrast': '0.15', '--num-workers': '2', '--seed': '42', '--device': str(DEVICE)}.items():
        command.extend([flag, source_option(flag, default)])
    protocol = {'source_record_sha256': v2_audit.sha256(source_record), 'command': command,
                'manifest_sha256': hashlib.sha256(manifest_text.encode()).hexdigest(), 'script_sha256': v2_audit.sha256(ST_SCRIPT)}
    protocol_path = ST_BUNDLE / 'source_provenance.json'
    if protocol_path.exists():
        if json.loads(protocol_path.read_text()) != protocol:
            raise ValueError('Protokol single-task berubah. Pulihkan konfigurasi atau pakai ST_BUNDLE baru.')
    elif ST_BUNDLE.exists() and any(ST_BUNDLE.iterdir()):
        raise FileExistsError('Folder berisi file tanpa provenance. Gunakan ST_BUNDLE baru.')
    if st_manifest.exists() and st_manifest.read_text() != manifest_text:
        raise ValueError('Manifest single-task berubah; jangan timpa hasil lama.')
    ST_BUNDLE.mkdir(parents=True, exist_ok=True)
    if not protocol_path.exists():
        protocol_path.write_text(json.dumps(protocol, indent=2), encoding='utf-8')
    if not st_manifest.exists():
        st_manifest.write_text(manifest_text, encoding='utf-8')
    if (ST_RUN / 'last_checkpoint.pt').exists():
        command.append('--resume')
    if (ST_RUN / 'metrics.json').exists():
        print('Run selesai; membaca hasil, bukan training ulang:', ST_RUN)
    else:
        print('Single-task image-only; no test. Output:', ST_RUN, flush=True)
        with (ST_BUNDLE / 'training.log').open('a', encoding='utf-8') as log:
            with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
                for line in process.stdout:
                    print(line, end='', flush=True)
                    log.write(line)
                    log.flush()
                return_code = process.wait()
            if return_code:
                raise subprocess.CalledProcessError(return_code, command)
else:
    print('Single-task training OFF. Aktifkan RUN_SINGLE_TASK=True jika siap; jangan jalankan bersamaan dengan proposed.')

In [ ]:
# Ringkasan validation: tidak melatih dan tidak membuka test.
st_metrics = ST_RUN / 'metrics.json'
if st_metrics.is_file():
    st_result = json.loads(st_metrics.read_text())
    best = min(st_result['history'], key=lambda r: r['val_loss'])
    comparison = [{'model': 'ResNet50 single-task', 'epoch': best['epoch'], 'val_accuracy': best['val_acc'],
                   'val_kappa': best['val_kappa'], 'criterion': 'validation CE minimum'}]
    proposed_path = OUTPUT_A / 'metrics.json'
    if proposed_path.is_file():
        proposed_result = json.loads(proposed_path.read_text())
        best_proposed = min(proposed_result['history'], key=lambda r: r['val_loss'])
        comparison.append({'model': 'Proposed augmented Step 11', 'epoch': best_proposed['epoch'],
                           'val_accuracy': best_proposed['val_acc'], 'val_kappa': best_proposed['val_kappa'],
                           'criterion': 'total multi-task validation loss minimum'})
    display(pd.DataFrame(comparison))
    display(pd.DataFrame(st_result['validation_metrics']['per_class']))
    print('Confusion matrix validation:', st_result['validation_metrics']['confusion_matrix'])
    print('Macro-F1 validation single-task:', st_result['validation_metrics']['macro_f1'])
    print('Dice/IoU single-task: tidak berlaku. Test tidak dievaluasi. Objective berbeda; bukan ablasi segmentasi terisolasi.')
else:
    print('Belum ada hasil single-task final. History sementara ada di history_partial.json jika epoch sudah selesai.')